# Figure 4 and SI 12

This notebook builds only two manuscript figures:

1. **FIG4**: graded depth profiles against slab DFT and free-tensor models,
   with the ``REC_L`` MD structure strip.
2. **SI12**: GIWAXS linecuts and peak spacings compared with the simulated
   structure drawn in FIG4 panel (d).

Wide-plate (183 mm) layout for FIG4. Left, a Q waterfall: 283.7 eV on the full
Q axis, and 284.2, 285.1, 287, and 289 eV decade-offset on a short-Q window
stretched by 2. Gradient (red) and the DFT slab (black, dashed) only. s-pol
is the darker circle and p-pol the lighter triangle, with the Fig. 2
reduced chi-square, $\chi^2_\nu = \chi^2/(N - n_{\mathrm{vary}})$, under
each model. Both models use the gradient fit's points, so $N$ matches. Right, one depth axis in nm carrying ``gamma(z)`` and
``rho(z)`` flanking a ``REC_L5c`` MD ball-and-stick cutout, each profile with
its fit +/- 2 sigma fill, closed at the
bottom by the Si bar. Graded gamma is Nevot-Croce smeared like the density.

The graded (red) curve is the sliced open multi-energy ``logY`` polish from
``bookended_multienergy_logY_polish_open_sliced/`` (backgrounds fixed at 0;
FIG4 outliers removed; film and substrate densities held at the open polish).
DFT-slab curves come from ``refit-refloxide.pkl`` via ``utils.refloxide_view``.
Free-tensor remains the pyref ``GlobalObjective`` pickle for now.


## 1. Load data


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erf

from utils import (
    models_fitting_results,
    models_root,
    read_fit,
    read_xrr,
)
from utils.graded_objective import (
    extract_graded_fit_context,
    load_bookended_multienergy_logY_polish,
)
from utils.helpers.fitting_helper import rxr
from utils.helpers.plotting_helper import set_plotting_defaults
from utils.models import configure_refloxide_fitting
from utils.refloxide_view import load_refloxide_fit, objective_view_at

import warnings

configure_refloxide_fitting()
set_plotting_defaults()

ENERGY = 283.7
# The accepted ML-DFT slab fit (also used by FIG3). The E_RES re-polish in
# dft_slab_eres_logY_polish/fit.pkl flipped the s/p labels before fitting and
# is not used.
REFLOXIDE_FIT_PATH = models_root / "xrr/znpc/refloxide/refit-refloxide.pkl"
# REFLOXIDE_FIT_PATH = (
#     models_fitting_results / "dft_slab_eres_logY_polish" / "fit.pkl"
# )

data = read_xrr("reflectivity_data", material="znpc", source="hub")
graded_polish = load_bookended_multienergy_logY_polish(
    models_fitting_results / "bookended_multienergy_logY_polish_open_sliced" / "fit.pkl"
)
graded_obj = graded_polish.at(ENERGY)
print(f"graded polish: {graded_polish.path.name}  protocol={graded_polish.summary.get('protocol')}")
free_fit = read_fit("free/free_en_offset_init_2.pkl", material="znpc", source="local")


def pick_energy_objective(global_obj, energy):
    """Return the single-energy objective nearest ``energy`` from a GlobalObjective."""
    energies = np.array([o.model.energy for o in global_obj.objectives], dtype=float)
    idx = int(np.argmin(np.abs(energies - energy)))
    return global_obj.objectives[idx]


if not REFLOXIDE_FIT_PATH.exists():
    raise FileNotFoundError(
        f"missing {REFLOXIDE_FIT_PATH} -- run "
        "notebooks/fitting/_dft_slab_eres_logY_polish.py first"
    )

refloxide_objective = load_refloxide_fit(REFLOXIDE_FIT_PATH)
# Hub read_xrr and refloxide share s/p labels; do not swap_pol.
refloxide_obj = objective_view_at(refloxide_objective, ENERGY, swap_pol=False)
dft_obj = refloxide_obj
free_obj = pick_energy_objective(free_fit, ENERGY)
dataset = data[str(ENERGY)]


## 2. Best-fit parameters

Graded book-ended film, oxide, and substrate at the probe energy (the red
curve in FIG4). Angles are stored in radians; the table also reports degrees.
Uncertainties are 1σ Hessian stderr when the parameter varied.


In [ ]:
import pandas as pd
import numpy as np
from uncertainties import ufloat

from utils.graded_objective import BOOKENDED_FILM_PARAM_NAMES

_ANGLE_PARAMS = ("alpha_bulk", "alpha_si", "alpha_vac")
_FILM_UNITS = {
    "total_thick": "Å",
    "surface_roughness": "Å",
    "tau_si": "Å",
    "tau_vac": "Å",
    "alpha_bulk": "rad",
    "alpha_si": "rad",
    "alpha_vac": "rad",
    "density_bulk": "g/cm³",
    "density_si": "g/cm³",
    "density_vac": "g/cm³",
    "energy_offset": "eV",
}


def _fmt_param(param, *, as_degrees: bool = False) -> str:
    value = float(param.value)
    err = getattr(param, "stderr", None)
    err_f = float(err) if err is not None else float("nan")
    if as_degrees:
        value = float(np.degrees(value))
        err_f = float(np.degrees(err_f)) if np.isfinite(err_f) else err_f
    if param.vary and np.isfinite(err_f) and err_f > 0:
        return f"{ufloat(value, err_f):.2uP}"
    if param.vary:
        return f"{value:g} (vary, no stderr)"
    return f"{value:g} (fixed)"


def _structure_param_table(objective) -> pd.DataFrame:
    vacuum, film, oxide, substrate = objective.model.structure.components
    _ = vacuum
    rows: list[dict[str, object]] = []

    film_names = tuple(n for n in BOOKENDED_FILM_PARAM_NAMES if n != "energy_offset")
    for name in film_names:
        if not hasattr(film, name):
            continue
        param = getattr(film, name)
        row = {
            "layer": "film",
            "parameter": name,
            "unit": _FILM_UNITS.get(name, ""),
            "value": _fmt_param(param),
            "vary": bool(param.vary),
        }
        if name in _ANGLE_PARAMS:
            row["value_deg"] = _fmt_param(param, as_degrees=True)
        else:
            row["value_deg"] = "-"
        rows.append(row)

    for layer_name, layer in (("oxide", oxide), ("substrate", substrate)):
        for pname, unit, param in (
            ("thick", "Å", layer.thick),
            ("rough", "Å", layer.rough),
            ("density", "g/cm³", layer.sld.density),
        ):
            rows.append(
                {
                    "layer": layer_name,
                    "parameter": pname,
                    "unit": unit,
                    "value": _fmt_param(param),
                    "value_deg": "-",
                    "vary": bool(param.vary),
                }
            )

    e_off = objective.model.energy_offset
    rows.append(
        {
            "layer": "model",
            "parameter": "energy_offset",
            "unit": "eV",
            "value": _fmt_param(e_off),
            "value_deg": "-",
            "vary": bool(e_off.vary),
        }
    )
    return pd.DataFrame(rows)


structure_params = _structure_param_table(graded_obj)
print(
    f"Graded structure @ {ENERGY} eV  "
    f"(protocol={graded_polish.summary.get('protocol')})"
)
for layer in ("film", "oxide", "substrate", "model"):
    block = structure_params.query("layer == @layer")
    if block.empty:
        continue
    print(f"\n{layer}")
    display(
        block.drop(columns=["layer"]).reset_index(drop=True).style.format(
            {"value": lambda x: x, "value_deg": lambda x: x},
            na_rep="-",
        )
    )


## 3. Helpers


In [ ]:
def nevot_croce_density(film, *, n_points: int = 2000, n_sigma: int = 5):
    """Convolve the book-ended density with the surface (vacuum) Névot-Croce term.

    The Névot-Croce interface is an error function, so its real-space density is
    the analytic profile convolved with a Gaussian of standard deviation equal to
    ``film.surface_roughness``. Vacuum (zero density) bounds the surface side and
    the film-edge density bounds the substrate side so the convolution sees the
    physical boundary on each end.

    Parameters
    ----------
    film
        Book-ended film component.
    n_points
        Samples across the film thickness.
    n_sigma
        Padding width in units of the surface roughness.

    Returns
    -------
    depth, rho_sharp, rho_rough
        Depth (angstrom) and the analytic and roughness-smeared densities
        (g/cm^3) on ``[0, total_thick]``.
    """
    total = float(film.total_thick.value)
    sigma = float(film.surface_roughness.value)
    depth = np.linspace(0.0, total, n_points)
    rho_sharp = np.asarray(film.local_density(depth), dtype=float)
    if sigma <= 0.0:
        return depth, rho_sharp, rho_sharp.copy()

    dz = depth[1] - depth[0]
    pad = int(np.ceil(n_sigma * sigma / dz))
    z_ext = np.concatenate(
        [depth[0] + dz * np.arange(-pad, 0), depth, depth[-1] + dz * np.arange(1, pad + 1)]
    )
    rho_ext = np.empty_like(z_ext)
    below, above = z_ext < 0.0, z_ext > total
    inside = ~(below | above)
    rho_ext[below] = 0.0
    rho_ext[inside] = np.asarray(film.local_density(z_ext[inside]), dtype=float)
    rho_ext[above] = float(film.local_density(total))

    kernel = np.exp(-0.5 * (dz * np.arange(-pad, pad + 1) / sigma) ** 2)
    kernel /= kernel.sum()
    rho_rough = np.convolve(rho_ext, kernel, mode="same")[pad : pad + n_points]
    return depth, rho_sharp, rho_rough


def nevot_croce_orientation(film, depth, rho_sharp, *, n_sigma: int = 5):
    """Density-weighted roughness smear of book-ended gamma (degrees).

    Convolves ``rho`` and ``rho * cos^2(gamma)`` with the same Gaussian used by
    ``nevot_croce_density``, vacuum-padded with zero density, then returns
    ``arccos sqrt( (rho cos^2)_s / rho_s )``. Matches the optical ``gamma_eff``
    convention and the slab panel's roughness presentation.
    """
    total = float(film.total_thick.value)
    sigma = float(film.surface_roughness.value)
    depth = np.asarray(depth, dtype=float)
    rho_sharp = np.asarray(rho_sharp, dtype=float)
    gamma_rad = np.asarray(film.orientation(depth), dtype=float)
    if sigma <= 0.0:
        return np.degrees(gamma_rad)
    dz = float(depth[1] - depth[0])
    pad = int(np.ceil(n_sigma * sigma / dz))
    z_ext = np.concatenate(
        [depth[0] + dz * np.arange(-pad, 0), depth, depth[-1] + dz * np.arange(1, pad + 1)]
    )
    below, above = z_ext < 0.0, z_ext > total
    inside = ~(below | above)
    rho_ext = np.empty_like(z_ext)
    c2_ext = np.empty_like(z_ext)
    rho_ext[below] = 0.0
    c2_ext[below] = 0.0
    rho_ext[inside] = np.asarray(film.local_density(z_ext[inside]), dtype=float)
    g_in = np.asarray(film.orientation(z_ext[inside]), dtype=float)
    c2_ext[inside] = rho_ext[inside] * np.cos(g_in) ** 2
    rho_edge = float(film.local_density(total))
    g_edge = float(film.orientation(np.array([total]))[0])
    rho_ext[above] = rho_edge
    c2_ext[above] = rho_edge * np.cos(g_edge) ** 2
    kernel = np.exp(-0.5 * (dz * np.arange(-pad, pad + 1) / sigma) ** 2)
    kernel /= kernel.sum()
    rho_s = np.convolve(rho_ext, kernel, mode="same")[pad : pad + depth.size]
    c2_s = np.convolve(c2_ext, kernel, mode="same")[pad : pad + depth.size]
    ratio = np.clip(np.where(rho_s > 1e-12, c2_s / np.maximum(rho_s, 1e-12), 0.0), 0.0, 1.0)
    return np.degrees(np.arccos(np.sqrt(ratio)))


def slab_film_profiles(model, idx=(1, 2, 3), n=600):
    """Erf-smoothed orientation (deg) and density for the 3-slab ZnPc film.

    Measured from the vacuum interface so the depth axis matches the graded film.
    Assumes vacuum|surf|bulk|inter|SiO2|Si and that component[i].rough sets the
    interface above slab i. The vacuum|surf interface (``component[idx[0]].rough``)
    multiplies the density by the same erf step the graded profile gets from its
    surface Nevot-Croce convolution, so both models start from zero density in
    vacuum. Orientation is a density-weighted quantity and is unaffected by that
    step.
    """
    s = [model.structure[i] for i in idx]
    t = np.array([float(c.thick.value) for c in s])
    a = np.array([float(c.sld.rotation.value) for c in s])
    rho = np.array([float(c.sld.density.value) for c in s])
    r0 = max(float(s[0].rough.value), 1e-6)
    r1 = max(float(s[1].rough.value), 1e-6)
    r2 = max(float(s[2].rough.value), 1e-6)
    total = float(t.sum())
    z = np.linspace(0.0, total, n)
    z1, z2 = t[0], t[0] + t[1]
    step0 = 0.5 * (1.0 + erf(z / (np.sqrt(2.0) * r0)))
    step1 = 0.5 * (1.0 + erf((z - z1) / (np.sqrt(2.0) * r1)))
    step2 = 0.5 * (1.0 + erf((z - z2) / (np.sqrt(2.0) * r2)))

    def blend(v):
        return v[0] + (v[1] - v[0]) * step1 + (v[2] - v[1]) * step2

    return z, np.degrees(blend(a)), step0 * blend(rho)


QMIN = 0.1


def log_residual(y, y_model, y_err):
    """Normalized log difference ``(ln y - ln y_model) / (y_err / y)``.

    Matches the fractional-error log residual used in the edge-fit helpers
    (``wm = y_err / y``). Prefer this over ``(y - y_model) / y_err`` on
    log-scaled reflectivity, where absolute residuals overweight low-R points.
    """
    y = np.asarray(y, dtype=float)
    y_model = np.asarray(y_model, dtype=float)
    y_err = np.asarray(y_err, dtype=float)
    return (np.log(np.clip(y, 1e-30, None)) - np.log(np.clip(y_model, 1e-30, None))) / (
        y_err / y
    )


def rms_residual(model, pol=("s", "p")):
    """RMS normalized log residual over the plotted q window."""
    chis = []
    for p in pol:
        d = getattr(dataset, p)
        m = d.x >= QMIN
        chis.append(log_residual(d.y[m], rxr(d.x[m], model, p), d.y_err[m]))
    return float(np.sqrt(np.mean(np.concatenate(chis) ** 2)))


## 4. Profiles and painters


In [ ]:
from matplotlib.lines import Line2D

C_S = C_P = "0.45"
C_GRADED = "red"
C_DFT = "k"
C_FREE = "0.42"
MODELS = [
    {"name": "Graded", "obj": graded_obj, "color": C_GRADED, "ls": "-", "lw": 1.1, "z": 5},
    {"name": "DFT slab", "obj": dft_obj, "color": C_DFT, "ls": "--", "lw": 1.0, "z": 6},
    {"name": "Free slab", "obj": free_obj, "color": C_FREE, "ls": "-", "lw": 1.0, "z": 4},
]

Q = np.linspace(QMIN, float(dataset.s.x.max()), 1500)
DATA_EVERY = 1
Z_LEFT = -6.0
R_YLIM = (1e-8, 1e-3)
GAMMA_YLIM = (20, 75)
RHO_YLIM = (0.5, 2.2)
CHI_YLIM = (-6.0, 6.0)
TAU_COLOR = "#0b7f8a"
TAU_LW = 0.85
TAU_LS = (0, (4, 3))

film = extract_graded_fit_context(graded_obj).film
TOTAL = float(film.total_thick.value)
TAU_VAC, TAU_SI = float(film.tau_vac.value), float(film.tau_si.value)
DEPTH_XLIM = (Z_LEFT, TOTAL)
Z_G, RHO_SHARP, RHO_ROUGH = nevot_croce_density(film)
ALPHA_G = np.degrees(np.asarray(film.orientation(Z_G), dtype=float))
ALPHA_ROUGH = nevot_croce_orientation(film, Z_G, RHO_SHARP)
C_MD = "#007681"
Z_DFT, ALPHA_DFT, RHO_DFT = slab_film_profiles(dft_obj.model)
Z_FREE, ALPHA_FREE, RHO_FREE = slab_film_profiles(free_obj.model)
RMSE = {m["name"]: rms_residual(m["obj"].model) for m in MODELS}


def _subsample_xyerr(x, y, yerr, every):
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    err_arr = np.asarray(yerr, dtype=float)
    if every <= 1:
        return x_arr, y_arr, err_arr
    idx = np.arange(0, len(x_arr), every, dtype=int)
    return x_arr[idx], y_arr[idx], err_arr[idx]


def highlight_interface(ax, *, halo=True, labels=True):
    """Vacuum and buried graded-interface markers (fig_2 diagnostic tau styling)."""
    for z_tau, txt, ha, dx in (
        (TAU_VAC, r"$\tau_\mathrm{vac}$", "left", 3),
        (TOTAL - TAU_SI, r"$\tau_\mathrm{Si}$", "right", -3),
    ):
        if halo:
            ax.axvline(
                z_tau,
                color="white",
                lw=TAU_LW + 1.1,
                solid_capstyle="butt",
                zorder=4,
            )
        ax.axvline(
            z_tau,
            color=TAU_COLOR,
            lw=TAU_LW,
            ls=TAU_LS,
            solid_capstyle="butt",
            zorder=5,
        )
        if labels:
            ax.annotate(
                txt,
                xy=(z_tau, 0.5),
                xycoords=("data", "axes fraction"),
                xytext=(dx, 0),
                textcoords="offset points",
                va="top",
                ha=ha,
                color=TAU_COLOR,
            )


def draw_reflectivity(ax, legend_loc="lower left"):
    for pol, marker in (("s", "o"), ("p", "^")):
        d = getattr(dataset, pol)
        m = d.x >= QMIN
        xs, ys, es = _subsample_xyerr(d.x[m], d.y[m], d.y_err[m], DATA_EVERY)
        col = C_S if pol == "s" else C_P
        ax.errorbar(
            xs,
            ys,
            yerr=es,
            fmt=marker,
            lw=0,
            ms=3.2,
            elinewidth=0.5,
            capsize=1.3,
            capthick=0.5,
            color=col,
            ecolor=col,
            zorder=10,
        )
    for mdl in MODELS:
        for pol in ("s", "p"):
            ax.plot(
                Q,
                rxr(Q, mdl["obj"].model, pol),
                color=  mdl["color"],
                ls=mdl["ls"],
                lw=mdl["lw"],
                zorder=mdl["z"],
                label=mdl["name"] if pol == "s" else None,
            )
    ax.set_yscale("log")
    ax.set_xlim(QMIN, Q.max())
    ax.set_ylim(*R_YLIM)
    ax.set_ylabel("Reflectivity (a.u.)")
    ax.minorticks_on()
    ax.legend(
        loc=legend_loc,
        # fontsize=6.5,
        handlelength=1.2,
        handletextpad=0.4,
        borderpad=0.3,
        labelspacing=0.25,
        framealpha=0.9,
        edgecolor="0.75",
        fancybox=False,
    )


def draw_residual(ax):
    yl, yh = CHI_YLIM
    ax.axhline(
        0.0,
        color=plt.rcParams["axes.edgecolor"],
        lw=plt.rcParams["axes.linewidth"],
    )
    d = dataset.p
    m = d.x >= QMIN
    residual_order = [mdl for mdl in MODELS if mdl["name"] != "DFT slab"] + [
        mdl for mdl in MODELS if mdl["name"] == "DFT slab"
    ]
    for mdl in residual_order:
        chi = log_residual(d.y[m], rxr(d.x[m], mdl["obj"].model, "p"), d.y_err[m])
        ax.plot(
            d.x[m],
            chi,
            color=mdl["color"],
            ls=mdl["ls"],
            lw=0.7 if mdl["name"] == "DFT slab" else 0.6,
            zorder=6 if mdl["name"] == "DFT slab" else mdl["z"],
            solid_capstyle="butt",
            dash_capstyle="butt",
        )
    ax.set_ylim(yl, yh)
    ax.set_yticks([-4, 0, 4])
    ax.set_ylabel(r"$\Delta\ln Ref./\sigma$")
    ax.set_xlim(QMIN, Q.max())
    ax.set_xlabel(r"$Q\ (\mathrm{\AA}^{-1})$")
    rmse_label = Line2D([], [], color="none", label="RMSE:")
    handles = [
        rmse_label,
        *[
            Line2D(
                [],
                [],
                color=mdl["color"],
                ls=mdl["ls"],
                lw=1.0,
                solid_capstyle="butt",
                dash_capstyle="butt",
                label=f"{RMSE[mdl['name']]:.1f}",
            )
            for mdl in MODELS
        ],
    ]

    class _EmptyHandle:
        def legend_artist(self, legend, orig_handle, fontsize, handlebox):
            return None

    ax.legend(
        handles=handles,
        loc="lower left",
        bbox_to_anchor=(-0.02, 0.04),
        bbox_transform=ax.transAxes,
        ncols=len(handles),
        # fontsize=6.0,
        handlelength=1,
        handletextpad=0.25,
        columnspacing=0.45,
        borderpad=0.25,
        borderaxespad=0.0,
        labelspacing=0.2,
        framealpha=0.9,
        edgecolor="0.75",
        fancybox=False,
        handler_map={rmse_label: _EmptyHandle()},
    )


def draw_orientation(ax, tau_labels=True):
    ax.plot(Z_FREE, ALPHA_FREE, color=C_FREE, ls="-", lw=0.9, zorder=3)
    ax.plot(Z_DFT, ALPHA_DFT, color=C_DFT, ls="--", lw=0.9, zorder=3)
    ax.plot(Z_G, ALPHA_ROUGH, color=C_GRADED, lw=1.1, zorder=5)
    highlight_interface(ax, halo=True, labels=tau_labels)
    ax.set_ylabel(r"$\gamma$ ($\circ$)")
    ax.set_ylim(*GAMMA_YLIM)
    ax.set_xlim(*DEPTH_XLIM)


def draw_density(ax, key=True):
    ax.plot(Z_FREE, RHO_FREE, color=C_FREE, ls="-", lw=0.9, zorder=3)
    ax.plot(Z_DFT, RHO_DFT, color=C_DFT, ls="--", lw=0.9, zorder=3, label="Slab")
    surf = Z_G <= TAU_VAC
    ax.plot(
        Z_G[surf],
        RHO_SHARP[surf],
        color=C_GRADED,
        ls=":",
        lw=0.8,
        zorder=4,
    )
    ax.plot(Z_G, RHO_ROUGH, color=C_GRADED, lw=1.0, zorder=5, label="Gradient")
    highlight_interface(ax, halo=True, labels=False)
    ax.set_ylabel(r"$\rho$ (g/cm$^{3}$)")
    ax.set_xlim(*DEPTH_XLIM)
    ax.set_ylim(*RHO_YLIM)
    ax.set_yticks([0.5, 1, 1.5, 2])
    if key:
        ax.legend(loc="lower left", frameon=False, fontsize=6.5, handlelength=1.4)


def shade_substrate(
    ax,
    z0=None,
    z1=None,
    color="0.40",
    alpha_max=0.22,
    power=2.0,
    n=80,
    label=None,
    label_color="0.35",
):
    """Grey gradient that fades in toward the substrate side of the panel."""
    if z1 is None:
        z1 = TOTAL
    if z0 is None:
        z0 = TOTAL - 2.0 * TAU_SI
    edges = np.linspace(z0, z1, n + 1)
    alphas = alpha_max * np.linspace(0.0, 1.0, n) ** power
    for x0, x1, a in zip(edges[:-1], edges[1:], alphas):
        ax.axvspan(x0, x1, color=color, alpha=float(a), lw=0, zorder=0)
    if label:
        ax.annotate(
            label,
            xy=(z1, 0.03),
            xycoords=("data", "axes fraction"),
            xytext=(-2, 0),
            textcoords="offset points",
            ha="right",
            va="bottom",
            color=label_color,
        )


## 5. Figure 4

Writes ``FIG4.pdf`` / ``FIG4.png``. The structure column is a disk cutout from
the ``REC_L5c`` LAMMPS reconstruction (``fig4_md_strip.py``); atomistic molecules
are deferred. Graded ``gamma`` is Névot-Croce smeared like the density. MD
``gamma_eff`` and ``rho`` overlay the fit curves. Panel letters are off while
spacing is settled.


In [ ]:
# ---------------------------------------------------------------------------
# Wide plate. Left half, reflectivity and residual. Right half, gamma(z) and
# rho(z) flanking the structure strip with no gutters between them, closed by
# the Si bar. DEPTH_ORIENT swings the whole right half between a vertical
# depth axis (vacuum up) and a horizontal one (the original FIG4 convention).
# ---------------------------------------------------------------------------
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.ticker import MaxNLocator
from matplotlib.transforms import Affine2D

_MS_DIR = Path.cwd().resolve()
if not (_MS_DIR / "fig4_panel_b_growth.py").is_file():
    _alt = _MS_DIR / "notebooks" / "manuscript"
    if (_alt / "fig4_panel_b_growth.py").is_file():
        _MS_DIR = _alt.resolve()
if not (_MS_DIR / "fig4_panel_b_growth.py").is_file():
    raise FileNotFoundError(
        "fig4_panel_b_growth.py not found beside the notebook or under "
        "notebooks/manuscript relative to the kernel cwd"
    )
if str(_MS_DIR) not in sys.path:
    sys.path.insert(0, str(_MS_DIR))

import fig4_panel_b_growth as _fig41

importlib.reload(_fig41)

# Substrate treatment for the molecular strip. The reload above resets the
# module default, so the flag has to be set here rather than in an earlier
# cell. See the comparison cell at the end of the notebook.
#   "off"   published behaviour, the acceptance test is on the molecular centre
#   "clip"  reject a molecule whose footprint crosses into the substrate
#   "shift" the same, with the lamellar register seeded where a molecule fits
#   "tilt"  lay the offending molecule flatter until it clears, deleting none
SUBSTRATE_MODE = "off"
SUB_CLEAR = 1
_fig41.SUBSTRATE_MODE = SUBSTRATE_MODE
_fig41.SUB_CLEAR = SUB_CLEAR
_fig41.SEED = 10
_fig41.PACKING = "crystal"  # pi spacing and column herringbone of the relaxed cell; "main" restores the old strip

# Panel (c) strip settings. These restate the module defaults so the figure does not
# depend on them silently; change them here, not in fig4_panel_b_growth.py.
#   CLASH_A           no two rings of one column closer than this (atom/bond centres, A)
#   SUB_ROUGH_A       rms reach below the mean Si surface (oxide roughness from the fit)
#   SETTLE_ZONE_TAU   lamellae within this many tau_Si of the Si settle onto the one below
#   FILM_DENSE        settled lamellae keep the crystal pitch, so the film has no voids
#   DENSITY_CORRECT   remove the resulting excess density from the partial contact covering
#   CONTACT_KEEP_MIN  smallest fraction of the contact covering that is kept
#   DRAW_BEHIND, BEHIND_GREY, BEHIND_ALPHA   column behind (opposite herringbone class)
_fig41.CLASH_A = 1.8
_fig41.SUB_ROUGH_A = 2.7
#   SURFACE_VACANCIES   broken free-surface terrace in the front column (False fills it)
_fig41.SURFACE_VACANCIES = True  # leaves out SURFACE_VACANCY_FRAC (0.15) of the top terrace
_fig41.SETTLE_ZONE_TAU = 10.0
_fig41.FILM_DENSE = False  # settled lamellae dilate with rho(z), so the filling layer is not overpacked
_fig41.DENSITY_CORRECT = True
_fig41.CONTACT_KEEP_MIN = 0.55
#   PATCH_SEED          layout of the lying contact islands only (2 gives four islands of 2 to 3)
#   PATCH_GLOW          red halo marking the lying contact islands, drawn above the film
_fig41.PATCH_SEED = 2
_fig41.CONTACT_PATCH_PERIODS_A = (22.0, 33.0, 51.0)
_fig41.PATCH_GLOW = True
_fig41.PATCH_STYLE = "color"  # or "glow" (red halo)
_fig41.PATCH_PALETTE = "red"  # or "green"
#   CONTACT_PATCHES     partial contact covering as closed patches of lying molecules
#   CONTACT_TILT_DEG    tilt of those patches (more lying than standing)
_fig41.CONTACT_PATCHES = True
_fig41.CONTACT_TILT_DEG = 28.0
#   PATCH_SINK_A        how far the lying patches reach into the rough oxide (A)
_fig41.PATCH_SINK_A = 3.0
_fig41.DRAW_BEHIND = True
_fig41.BEHIND_GREY = "0.70"
_fig41.BEHIND_ALPHA = 0.25

# Any mode other than the published one writes to its own name, so an
# experiment cannot silently overwrite the manuscript figure.
STRUCTURE_STYLE = "growth"  # "growth" molecule strip (main branch) | "cutout" 3D slab | "ticks" tilt ticks | "wire" all molecules
SAVENAME = "FIG4" if STRUCTURE_STYLE == "growth" else f"FIG4_{STRUCTURE_STYLE}"
HC_EV_A = 12398.419843320026
A2NM = 0.1
C_MD = "#007681"

# ---- 0. layout switches ------------------------------------------------------
# DEPTH_ORIENT  "vertical"   depth runs down the page; gamma and rho sit side by
#                            side (shared y), MD structure to their right.
#               "horizontal" depth runs left to right (legacy PRL layout).
# SHOW_WAVE     draws the narrow probe-wave strip alongside the depth panels.
# FILM_ORDER    left-to-right panel order in the depth block.
DEPTH_ORIENT = "vertical"
SHOW_WAVE = False
FILM_ORDER = ("gamma", "structure", "rho") if STRUCTURE_STYLE == "growth" else ("gamma", "rho", "structure")
FILM_WIDTH = {"gamma": 1.00, "rho": 1.00, "structure": 2.30 if STRUCTURE_STYLE == "growth" else 1.45}
STRUCTURE_GAP = 0.0 if STRUCTURE_STYLE == "growth" else 0.22
STRUCTURE_AXIS_OFFSET_PT = 4.0
FILM_HEIGHT = {"structure": 1.30, "gamma": 1.00, "rho": 1.00}
WAVE_WIDTH, WAVE_GAP = 0.22, 0.06

HORIZ = DEPTH_ORIENT == "horizontal"

# ---- 1. profiles from the optical model -------------------------------------
_film = extract_graded_fit_context(graded_obj).film
Z_G, RHO_SHARP, RHO_ROUGH = nevot_croce_density(_film)
ALPHA_G = np.degrees(np.asarray(_film.orientation(Z_G), dtype=float))
ALPHA_ROUGH = nevot_croce_orientation(_film, Z_G, RHO_SHARP)
TOTAL = float(_film.total_thick.value)
TAU_VAC = float(_film.tau_vac.value)
TAU_SI = float(_film.tau_si.value)

Z_DFT = Z_DFT * (TOTAL / float(Z_DFT[-1]))
Z_FREE = Z_FREE * (TOTAL / float(Z_FREE[-1]))

# Vacuum headroom (angstrom) above depth 0: the cutout needs room for the
# roughness tail of the top lamellae, the tick strip only draws what is there.
Z_TOP = {"growth": -4.0, "ticks": -5.0}.get(STRUCTURE_STYLE, -21.0)
SUB_NM = 1.0 if STRUCTURE_STYLE == "growth" else 1.6  # solid substrate bar closing the depth panels
SPAN_DISP = TOTAL - Z_TOP
SLAB_T = 14.37 # 11.5 # 13.8
L_MOL = 14.95
LAT_FRAC = 0.955
WAVELENGTH_NM = (HC_EV_A / ENERGY) * A2NM

_fig41.L_MOL = 14.95     # H-inclusive in-plane diameter, tpznpc_0m.xyz
_fig41.LAT_FRAC = 0.955  # lamellar interdigitation, from GIWAXS

ZN_G, ZN_DFT, ZN_FREE = Z_G * A2NM, Z_DFT * A2NM, Z_FREE * A2NM
TOTAL_NM, TAU_VAC_NM, TAU_SI_NM = TOTAL * A2NM, TAU_VAC * A2NM, TAU_SI * A2NM

# ---- 1b. REC_L5c MD profiles (atomistic strip drawn live on axS below) --------
import fig4_md_strip as _mdstrip

importlib.reload(_mdstrip)
_MD_TAG = "REC_L6:initial"
_MD_PROF_NPZ = _MS_DIR / "FIG4_md_profiles.npz"
_need_md = not _MD_PROF_NPZ.is_file()
if not _need_md:
    _probe = np.load(_MD_PROF_NPZ)
    _need_md = (
        abs(float(_probe["total_thick"]) - TOTAL) > 0.05
        or str(_probe["tag"]) != _MD_TAG
    )
if _need_md:
    _z_md, _g_md, _r_md = _mdstrip.md_profiles(_MD_TAG, total_thick=TOTAL, nlast=10)
    np.savez_compressed(
        _MD_PROF_NPZ, z=_z_md, gamma=_g_md, rho=_r_md, total_thick=TOTAL, tag=_MD_TAG
    )
_MD = np.load(_MD_PROF_NPZ)
Z_MD = np.asarray(_MD["z"], dtype=float)
GAMMA_MD = np.asarray(_MD["gamma"], dtype=float)
RHO_MD = np.asarray(_MD["rho"], dtype=float)
ZN_MD = Z_MD * A2NM
_MD_OK = RHO_MD > 0.25
print(
    f"MD {_MD_TAG}: {int(_MD_OK.sum())} bins with rho>0.25, "
    f"gamma [{np.nanmin(GAMMA_MD[_MD_OK]):.1f}, {np.nanmax(GAMMA_MD[_MD_OK]):.1f}] deg"
)

# Depth limits in drawing order, so the vertical case reads vacuum first.
DEPTH_LIM = (
    (Z_TOP * A2NM, TOTAL_NM + SUB_NM) if HORIZ
    else (TOTAL_NM + SUB_NM, Z_TOP * A2NM)
)

_PACK = _fig41.configure_packing(
    _film,
    z_left=Z_LEFT,
    z_g=Z_G,
    rho_sharp=RHO_SHARP,
    alpha_g=ALPHA_ROUGH,  # the same gamma(z) panel (b) plots, so the strip follows it lamella by lamella
    slab_thick=SLAB_T,
    highlight_fn=lambda ax, *, halo=True: None,  # rules drawn after placement
    xyz_path=_MS_DIR / "tpznpc_0m.xyz",
)
print(
    f"structure: slab {_PACK['slab_thick']:.1f} A, "
    f"bulk cell {_PACK['cell_area']:.1f} A^2/molecule, "
    f"median gamma {_PACK['gamma_median']:.1f} deg, "
    f"lambda {WAVELENGTH_NM:.2f} nm over {TOTAL_NM:.1f} nm of film"
)


# ---- 2. orientation-aware primitives (depth in nm) ---------------------------
# Every painter below draws in (depth, value) and lets these four helpers map
# the pair onto the axes, so one set of code serves both orientations.
def dv(depth, value):
    return (depth, value) if HORIZ else (value, depth)


def depth_rule(ax, z, **kw):
    (ax.axvline if HORIZ else ax.axhline)(z, **kw)


def depth_span(ax, z0, z1, **kw):
    (ax.axvspan if HORIZ else ax.axhspan)(z0, z1, **kw)


def set_depth_lim(ax):
    (ax.set_xlim if HORIZ else ax.set_ylim)(*DEPTH_LIM)


def set_value_lim(ax, lo, hi):
    (ax.set_ylim if HORIZ else ax.set_xlim)(lo, hi)


def set_value_label(ax, text):
    (ax.set_ylabel if HORIZ else ax.set_xlabel)(text)


def value_axis(ax):
    return ax.yaxis if HORIZ else ax.xaxis


def axis_limits(arrays, step, clamp, pad=0.08):
    """Tight limits around the plotted curves, rounded and clamped.

    The FIG4 constants were chosen for a wide horizontal panel and leave most
    of a narrow vertical panel empty, so the window is taken from the curves
    themselves and then clamped so it can never open wider than FIG4 used.
    """
    lo = min(float(np.min(a)) for a in arrays)
    hi = max(float(np.max(a)) for a in arrays)
    m = pad * (hi - lo)
    lo = np.floor((lo - m) / step) * step
    hi = np.ceil((hi + m) / step) * step
    return max(float(lo), clamp[0]), min(float(hi), clamp[1])


def tau_rules(ax):
    for z_tau in (TAU_VAC_NM, TOTAL_NM - TAU_SI_NM):
        depth_rule(ax, z_tau, color="white", lw=TAU_LW + 1.1,
                   solid_capstyle="butt", zorder=4)
        depth_rule(ax, z_tau, color=TAU_COLOR, lw=TAU_LW, ls=TAU_LS,
                   solid_capstyle="butt", zorder=5)


def tau_tags(ax):
    for z_tau, txt in ((TAU_VAC_NM, r"$\tau_\mathrm{vac}$"),
                       (TOTAL_NM - TAU_SI_NM, r"$\tau_\mathrm{Si}$")):
        if HORIZ:
            ax.annotate(txt, xy=(z_tau, 0.04),
                        xycoords=("data", "axes fraction"),
                        xytext=(2, 0), textcoords="offset points",
                        ha="left", va="bottom", color=TAU_COLOR, zorder=10)
        else:
            ax.annotate(txt, xy=(0.03, z_tau),
                        xycoords=("axes fraction", "data"),
                        xytext=(0, 2), textcoords="offset points",
                        ha="left", va="bottom", color=TAU_COLOR, zorder=10)


def zone_bands(ax):
    """Faint tint over the two book-ended zones, carried across every panel."""
    z_vac_edge = DEPTH_LIM[0] if HORIZ else DEPTH_LIM[1]
    depth_span(ax, z_vac_edge, TAU_VAC_NM, color=TAU_COLOR, alpha=0.05,
               lw=0, zorder=0)
    depth_span(ax, TOTAL_NM - TAU_SI_NM, TOTAL_NM, color=TAU_COLOR, alpha=0.05,
               lw=0, zorder=0)


# def substrate_bar(ax, label=False, zorder=1):
#     """Solid substrate bar closing the deep end of a depth panel.

#     ``zorder`` has to clear the molecules on the structure strip, but on the
#     profile panels it must sit BELOW the axis. ``axes.axisbelow`` is "line" in
#     the project style, which puts the axis (ticks included) at zorder 1.5, so a
#     bar drawn above that would swallow the inward ticks instead of letting them
#     read against the grey.
#     """
#     depth_span(ax, TOTAL_NM, TOTAL_NM + SUB_NM, color="0.55", lw=0,
#                zorder=zorder)
#     if label:
#         xy = ((TOTAL_NM + 0.5 * SUB_NM, 0.5) if HORIZ
#               else (0.5, TOTAL_NM + 0.5 * SUB_NM))
#         cc = ("data", "axes fraction") if HORIZ else ("axes fraction", "data")
#         ax.annotate("Si", xy=xy, xycoords=cc, ha="center", va="center",
#                     color="white", zorder=9, rotation=90 if HORIZ else 0)


SUB_SHADE = "grey"
SUB_ROUGH_NM = 1        # NOTE: nanometres, so 1 is 10 A; use 0.1 for 1 A
SUB_ALPHA = 1.00
SUB_ALPHA_TOP = 0.00
# tau_Si tint carried across the boundary and over the bar.
#   SUB_ZONE_FADE_NM  None no tint inside the bar, 0.0 full strength to the
#                     bottom, >0 decays with that length below the boundary
#   SUB_ZONE_ALPHA    alpha INSIDE the bar. It has to exceed the 0.05 used
#                     above the boundary: the same alpha over a mid grey gives
#                     only 53% of the colour shift it gives over white, so
#                     0.095 is the perceptually matched value.
SUB_ZONE_FADE_NM = 0.0
SUB_ZONE_ALPHA = 0.095


In [ ]:
import colorsys
import pickle

import matplotlib.patheffects as pe
from matplotlib.colors import to_rgb
from matplotlib.ticker import FixedLocator, NullFormatter, NullLocator
from scipy.special import erf

def substrate_bar(ax, label=False, zorder=1):
    """Substrate closing the deep end of a depth panel.

    Two layers. The grey is an opacity ramp confined to the bar, transparent at
    the top surface of the silicon and reaching SUB_ALPHA exactly at the bottom
    of the bar; the erf is divided by its own value there, so the colour shades
    to the full bottom whatever SUB_ROUGH_NM is set to. The tau_Si tint is then
    carried across the boundary and painted over the grey, so the interfacial
    band reads as running into the substrate instead of stopping at it.

    ``zorder`` has to clear the molecules on the structure strip, but on the
    profile panels it must sit BELOW the axis. ``axes.axisbelow`` is "line" in
    the project style, which puts the axis (ticks included) at zorder 1.5, so a
    bar drawn above that would swallow the inward ticks instead of letting them
    read against the grey. The tint goes at ``zorder + 0.1``, which clears the
    grey on both panel kinds and stays under the axis and the molecules.

    Both layers use a blended transform, spanning the whole panel in the value
    direction and data coordinates in depth, so they behave like ``depth_span``
    and do NOT require the value limits to have been set first. That matters:
    ``draw_orientation_v`` and ``draw_density_v`` both call this before
    ``set_value_lim``.
    """
    _n = 512
    _z0, _z1 = TOTAL_NM, TOTAL_NM + SUB_NM
    _zz = np.linspace(_z0, _z1, _n)

    def _paint(_rgb, _alpha, _zo):
        _rgba = np.empty((_n, 1, 4))
        _rgba[..., :3] = to_rgb(_rgb)
        _rgba[:, 0, 3] = np.clip(_alpha, 0.0, 1.0)
        if HORIZ:
            ax.imshow(_rgba.transpose(1, 0, 2), extent=(_z0, _z1, 0.0, 1.0),
                      origin="lower", aspect="auto", zorder=_zo,
                      interpolation="bilinear",
                      transform=ax.get_xaxis_transform())
        else:
            ax.imshow(_rgba, extent=(0.0, 1.0, _z1, _z0), origin="upper",
                      aspect="auto", zorder=_zo, interpolation="bilinear",
                      transform=ax.get_yaxis_transform())

    if SUB_ROUGH_NM <= 0.0:
        depth_span(ax, _z0, _z1, color=SUB_SHADE, alpha=SUB_ALPHA, lw=0,
                   zorder=zorder)
    else:
        _w = np.sqrt(2.0) * SUB_ROUGH_NM
        _f = erf((_zz - TOTAL_NM) / _w) / erf(SUB_NM / _w)
        _paint(SUB_SHADE, SUB_ALPHA_TOP + (SUB_ALPHA - SUB_ALPHA_TOP) * _f,
               zorder)

    if SUB_ZONE_FADE_NM is not None:
        _t = (np.full(_n, SUB_ZONE_ALPHA) if SUB_ZONE_FADE_NM <= 0.0
              else SUB_ZONE_ALPHA * np.exp(-(_zz - TOTAL_NM) / SUB_ZONE_FADE_NM))
        _paint(TAU_COLOR, _t, zorder + 0.1)

    if label:
        xy = ((TOTAL_NM + 0.5 * SUB_NM, 0.5) if HORIZ
              else (0.5, TOTAL_NM + 0.5 * SUB_NM))
        cc = ("data", "axes fraction") if HORIZ else ("axes fraction", "data")
        ax.annotate("Si", xy=xy, xycoords=cc, ha="center", va="center",
                    color="white", zorder=9, rotation=90 if HORIZ else 0)

def draw_wave(ax):
    """Incident wave at the probe wavelength, on the shared depth scale.

    Full extent of the depth axis, substrate bar included, with the phase
    origin at the Si interface so the node sits on the substrate and the wave
    is read out of it rather than out of the free surface.
    """
    z = np.linspace(min(DEPTH_LIM), max(DEPTH_LIM), 4000)
    ax.plot(*dv(z, np.sin(2.0 * np.pi * (TOTAL_NM - z) / WAVELENGTH_NM)),
            color=TAU_COLOR, lw=0.8, zorder=3)
    set_value_lim(ax, -1.15, 1.15)
    (ax.set_yticks if HORIZ else ax.set_xticks)([])
    for sp in (("left", "right", "top") if HORIZ else ("top", "right", "bottom")):
        ax.spines[sp].set_visible(False)
    # set_plotting_defaults turns on ticks along every spine (xtick.top,
    # ytick.right) and minor ticks with them. Hiding a spine does not hide its
    # ticks, so they have to go explicitly or they float in the gap before the
    # first profile. Same for the grid, which this strip does not need.
    ax.tick_params(which="both", bottom=not HORIZ, top=False,
                   right=False, left=HORIZ)
    ax.grid(False)


def vacuum_tail(z_neg):
    """Roughness-smeared density and gamma above the nominal surface (depth < 0).

    The same Gaussian smear as ``nevot_croce_density`` / ``nevot_croce_orientation``
    with vacuum (zero density) above depth 0, evaluated where those helpers stop,
    so panels (b) and (c) run up to the same depth as the structure in (d)
    instead of being cut at 0. Continuous with ``RHO_ROUGH`` and ``ALPHA_ROUGH`` at 0.
    """
    sig = float(film.surface_roughness.value)
    dz = Z_G[1] - Z_G[0]
    k = np.exp(-0.5 * ((np.asarray(z_neg)[:, None] - Z_G[None, :]) / sig) ** 2) * dz / (sig * np.sqrt(2.0 * np.pi))
    rho = k @ RHO_SHARP
    c2 = k @ (RHO_SHARP * np.cos(np.radians(ALPHA_G)) ** 2)
    gam = np.degrees(np.arccos(np.sqrt(np.clip(c2 / np.maximum(rho, 1e-12), 0.0, 1.0))))
    return rho, gam


Z_TAIL = np.linspace(min(Z_TOP, 0.0), 0.0, 60)
RHO_TAIL, ALPHA_TAIL = vacuum_tail(Z_TAIL)

_PROFILE_NAMES = (
    "total_thick",
    "surface_roughness",
    "tau_vac",
    "tau_si",
    "alpha_vac",
    "alpha_si",
    "alpha_bulk",
    "density_vac",
    "density_si",
    "density_bulk",
)


def _smeared_on_grid():
    """Smeared density and tilt on the vacuum tail followed by ``Z_G``."""
    depth, rho_sharp, rho_rough = nevot_croce_density(film, n_points=800)
    gamma = nevot_croce_orientation(film, depth, rho_sharp)
    sharp = np.asarray(film.local_density(Z_G), dtype=float)
    g_sharp = np.degrees(np.asarray(film.orientation(Z_G), dtype=float))
    sigma = float(film.surface_roughness.value)
    dz = float(Z_G[1] - Z_G[0])
    kernel = np.exp(-0.5 * ((Z_TAIL[:, None] - Z_G[None, :]) / sigma) ** 2)
    kernel *= dz / (sigma * np.sqrt(2.0 * np.pi))
    rho_tail = kernel @ sharp
    moment = kernel @ (sharp * np.cos(np.radians(g_sharp)) ** 2)
    gamma_tail = np.degrees(
        np.arccos(np.sqrt(np.clip(moment / np.maximum(rho_tail, 1e-12), 0.0, 1.0)))
    )
    rho = np.concatenate([rho_tail, np.interp(Z_G, depth, rho_rough)])
    gam = np.concatenate([gamma_tail, np.interp(Z_G, depth, gamma)])
    return rho, gam


def _profile_confidence_bands():
    """Pointwise +/- 2 sigma bands of the plotted gradient profiles.

    Uses the Laplace covariance of the sliced ``logY`` fit. Densities are
    included in the Hessian even though the optimizer held them fixed. The
    half-width at each depth is twice the delta-method standard deviation of
    the same Nevot-Croce smear as the drawn curves, and the band is centered
    on those curves. Gamma is given zero width where the plotted density is
    below 0.2 g/cm^3, because the tilt ratio is not meaningful in vacuum.

    Returns
    -------
    depth_nm, gamma_lo, gamma_hi, rho_lo, rho_hi : ndarray
        Depth in nm and the fit minus and plus 2 sigma, tail then film.
    """
    with graded_polish.path.open("rb") as handle:
        objective = pickle.load(handle)["gpu_objective"]
    held = []
    try:
        for parameter in objective.parameters.flattened():
            if parameter.name in ("density_vac", "density_si", "density_bulk") and not parameter.vary:
                parameter.vary = True
                held.append(parameter)
        covariance = np.asarray(objective.covar(), dtype=float)
        varying = list(objective.varying_parameters())
    finally:
        for parameter in held:
            parameter.vary = False
    index = {parameter.name: i for i, parameter in enumerate(varying)}
    columns = [index[name] for name in _PROFILE_NAMES]
    block = covariance[np.ix_(columns, columns)]
    block = 0.5 * (block + block.T)
    floor = float(np.linalg.eigvalsh(block).min())
    if floor < 0.0:
        block = block + (-floor + 1e-12) * np.eye(block.shape[0])
    stderr = np.sqrt(np.clip(np.diag(block), 0.0, None))
    original = {name: float(getattr(film, name).value) for name in _PROFILE_NAMES}
    jac_rho = []
    jac_gam = []
    try:
        for name, sigma in zip(_PROFILE_NAMES, stderr, strict=True):
            step = 0.1 * float(sigma) if sigma > 0.0 else 1e-6 * max(abs(original[name]), 1.0)
            getattr(film, name).value = original[name] + step
            rho_p, gam_p = _smeared_on_grid()
            getattr(film, name).value = original[name] - step
            rho_m, gam_m = _smeared_on_grid()
            getattr(film, name).value = original[name]
            jac_rho.append((rho_p - rho_m) / (2.0 * step))
            jac_gam.append((gam_p - gam_m) / (2.0 * step))
    finally:
        for name, value in original.items():
            getattr(film, name).value = value
    jac_rho = np.column_stack(jac_rho)
    jac_gam = np.column_stack(jac_gam)
    sig_rho = np.sqrt(np.clip(np.einsum("ni,ij,nj->n", jac_rho, block, jac_rho), 0.0, None))
    sig_gam = np.sqrt(np.clip(np.einsum("ni,ij,nj->n", jac_gam, block, jac_gam), 0.0, None))
    rho_ref = np.concatenate([RHO_TAIL, RHO_ROUGH])
    gamma_ref = np.concatenate([ALPHA_TAIL, ALPHA_ROUGH])
    sig_gam = np.where(rho_ref < 0.2, 0.0, sig_gam)
    depth_nm = np.concatenate([Z_TAIL, Z_G]) * A2NM
    mid = Z_G.size // 2
    print(
        "profile 2-sigma half-width at mid-film: "
        f"gamma {2.0 * sig_gam[-Z_G.size + mid]:.2f} deg, "
        f"rho {2.0 * sig_rho[-Z_G.size + mid]:.3f} g/cm^3"
    )
    return (
        depth_nm,
        gamma_ref - 2.0 * sig_gam,
        gamma_ref + 2.0 * sig_gam,
        rho_ref - 2.0 * sig_rho,
        rho_ref + 2.0 * sig_rho,
    )


Z_BAND_NM, GAMMA_LO, GAMMA_HI, RHO_LO, RHO_HI = _profile_confidence_bands()


def _confidence_band(ax, lo, hi):
    """Fill the fit +/- 2 sigma along the depth axis."""
    if HORIZ:
        ax.fill_between(Z_BAND_NM, lo, hi, color=C_GRADED, alpha=0.28, lw=0, zorder=3)
    else:
        ax.fill_betweenx(Z_BAND_NM, lo, hi, color=C_GRADED, alpha=0.28, lw=0, zorder=3)


def draw_orientation_v(ax):
    _confidence_band(ax, GAMMA_LO, GAMMA_HI)
    # ax.plot(*dv(ZN_DFT, ALPHA_DFT), color=C_DFT, ls="--", lw=0.75, zorder=3)
    ax.plot(*dv(ZN_G, ALPHA_ROUGH), color=C_GRADED, lw=0.75, zorder=5)
    ax.plot(*dv(Z_TAIL * A2NM, ALPHA_TAIL), color=C_GRADED, lw=0.75, zorder=5)
    zone_bands(ax)
    tau_rules(ax)
    substrate_bar(ax, zorder=1.2)
    set_value_lim(ax, *axis_limits([ALPHA_ROUGH, ALPHA_TAIL, GAMMA_LO, GAMMA_HI], 5.0, GAMMA_YLIM))
    value_axis(ax).set_major_locator(MaxNLocator(3, prune="both"))
    ax.minorticks_on()
    set_value_label(ax, r"$\gamma$ ($\circ$)")


def draw_density_v(ax):
    surf = Z_G <= TAU_VAC
    _confidence_band(ax, RHO_LO, RHO_HI)
    # ax.plot(*dv(ZN_DFT, RHO_DFT), color=C_DFT, ls="--", lw=0.75, zorder=3)
    ax.plot(*dv(ZN_G[surf], RHO_SHARP[surf]), color=C_GRADED, ls=":", lw=0.75,
            zorder=4)
    ax.plot(*dv(ZN_G, RHO_ROUGH), color=C_GRADED, lw=1.0, zorder=5)
    ax.plot(*dv(Z_TAIL * A2NM, RHO_TAIL), color=C_GRADED, lw=1.0, zorder=5)
    zone_bands(ax)
    tau_rules(ax)
    substrate_bar(ax, zorder=1.2)
    set_value_lim(ax, *axis_limits([RHO_ROUGH, RHO_TAIL, RHO_LO, RHO_HI], 0.1, RHO_YLIM))
    value_axis(ax).set_major_locator(MaxNLocator(3, prune="both"))
    ax.minorticks_on()
    set_value_label(ax, r"$\rho$ (g cm$^{-3}$)")


# ---- 2b. residuals ----------------------------------------------------------
# Toggle: "linear" -> (R - R_fit) / sigma; "log" -> (ln R - ln R_fit) / (sigma / R).
# RMSE is the RMS of chi over both polarizations (legend).
RESIDUAL_KIND = "linear"  # "linear" | "log"
from matplotlib.lines import Line2D

_RESIDUAL_YLABEL = {
    "linear": r"$Res./\sigma$",
    "log": r"$(\ln R - \ln R_{\mathrm{fit}})/(\sigma/R)$",
}
if RESIDUAL_KIND not in _RESIDUAL_YLABEL:
    raise ValueError(f"RESIDUAL_KIND must be one of {tuple(_RESIDUAL_YLABEL)}, got {RESIDUAL_KIND!r}")


def residual_at(model, pol):
    """x, chi, R_data, R_model, sigma over the plotted q window.

    ``chi`` follows ``RESIDUAL_KIND`` (linear or log-normalized).
    """
    d = getattr(dataset, pol)
    m = d.x >= QMIN
    x = np.asarray(d.x[m], dtype=float)
    r_data = np.asarray(d.y[m], dtype=float)
    sigma = np.asarray(d.y_err[m], dtype=float)
    r_model = np.asarray(rxr(x, model, pol), dtype=float)
    if RESIDUAL_KIND == "linear":
        chi = (r_data - r_model) / sigma
    else:
        chi = log_residual(r_data, r_model, sigma)
    return x, chi, r_data, r_model, sigma


RMSE = {}
print(f"residual kind: {RESIDUAL_KIND}")
print(f"{'model':<11}{'pol':>4}{'n':>5}{'bad':>5}{'med R_d/R_m':>13}"
      f"{'med |chi|':>11}{'rms chi':>10}")
for _mdl in MODELS:
    _chis = []
    for _pol in ("s", "p"):
        _x, _chi, _rd, _rm, _sg = residual_at(_mdl["obj"].model, _pol)
        _chis.append(_chi)
        _bad = int(np.count_nonzero(~np.isfinite(_rm)) + np.count_nonzero(_sg <= 0))
        print(f"{_mdl['name']:<11}{_pol:>4}{len(_x):>5}{_bad:>5}"
              f"{np.median(_rd / _rm):>13.3f}"
              f"{np.median(np.abs(_chi)):>11.2f}"
              f"{np.sqrt(np.mean(_chi ** 2)):>10.2f}")
    RMSE[_mdl["name"]] = float(np.sqrt(np.mean(np.concatenate(_chis) ** 2)))
print("RMSE:", {k: round(v, 1) for k, v in RMSE.items()})


def draw_residual_chi(ax):
    """Residual strip, styling and legend as in ``draw_residual``."""
    ax.axhline(
        0.0,
        color=plt.rcParams["axes.edgecolor"],
        lw=plt.rcParams["axes.linewidth"],
    )
    order = [m for m in MODELS if m["name"] != "DFT slab"] + [
        m for m in MODELS if m["name"] == "DFT slab"
    ]
    for mdl in order:
        x, chi, *_ = residual_at(mdl["obj"].model, "p")
        ax.plot(
            x,
            chi,
            color=mdl["color"],
            ls=mdl["ls"],
            lw=0.7 if mdl["name"] == "DFT slab" else 0.6,
            zorder=6 if mdl["name"] == "DFT slab" else 10,
            solid_capstyle="butt",
            dash_capstyle="butt",
        )
    ax.set_ylim(*CHI_YLIM)
    ax.set_yticks([-4, 0, 4])
    ax.set_ylabel(_RESIDUAL_YLABEL[RESIDUAL_KIND])
    ax.set_xlim(QMIN, Q.max())
    ax.set_xlabel(r"$Q\ (\mathrm{\AA}^{-1})$")
    rmse_label = Line2D([], [], color="none", label="RMSE:")
    handles = [
        rmse_label,
        *[
            Line2D(
                [],
                [],
                color=mdl["color"],
                ls=mdl["ls"],
                lw=1.0,
                solid_capstyle="butt",
                dash_capstyle="butt",
                label=f"{RMSE[mdl['name']]:.1f}",
            )
            for mdl in MODELS
        ],
    ]

    class _EmptyHandle:
        def legend_artist(self, legend, orig_handle, fontsize, handlebox):
            return None

    ax.legend(
        handles=handles,
        loc="lower left",
        bbox_to_anchor=(-0.02, 0.04),
        bbox_transform=ax.transAxes,
        ncols=len(handles),
        # fontsize=6.0,
        handlelength=1,
        handletextpad=0.25,
        columnspacing=0.45,
        borderpad=0.25,
        borderaxespad=0.0,
        labelspacing=0.2,
        framealpha=0.9,
        edgecolor="0.75",
        fancybox=False,
        handler_map={rmse_label: _EmptyHandle()},
    )


# ---- 2c. reflectivity, panel (a) ---------------------------------------------
# Graded against DFT slab. 283.7 eV stays on the main axes. The other resonant
# energies are decade-offset on a short-Q window stretched by STACK_Q_SCALE.

STACK_ENERGIES = (283.7, 284.2, 285.1, 287.0, 289.0)
STACK_MODELS = [m for m in MODELS if m["name"] in ("Graded", "DFT slab")]
STACK_LABELS = {"Graded": "Gradient", "DFT slab": "Slab"}  # legend text only
MAIN_Q_LO = 0.0
MAIN_YLIM = (1e-11, 1.5)
INSET_ENERGIES = STACK_ENERGIES[1:]  # 283.7 eV lives in the main axes only
INSET_BOUNDS = (0.05, 0.07, 0.47, 0.42)  # axes fraction, under the 283.7 eV curve; ~2.2x the main Q scale
LEGEND_ANCHOR = (0.86, 0.98)
N_PARAMS = {"Graded": 12, "DFT slab": 17}  # varying parameters of each model
FIT_SIGMA = 2.0  # a point is matched when the model is within this many error bars of it
INSET_Q = (0.001, 0.070)
STACK_Q_SCALE = 2.0
INSET_LABEL_Q, INSET_LABEL_GAP_DEC = 0.03, 0.0
INSET_GAP_DEC = 0.55
INSET_LABELS = False
INSET_LABEL_DX, INSET_LABEL_DY = -1.0, 2.0
INSET_EVERY = 1
STACK_DECADES = 2.45
STACK_TOP_DEC = 2.45
DATA_MS = 2.0

_MASK_Q = {
    (284.2, "p"): (0.01361, 0.01465, 0.01570),
    (284.2, "s"): (0.03449, 0.03656, 0.03760, 0.03863),
    (285.1, "p"): (0.04188,),
    (287.0, "p"): (0.068578, 0.070815),
}


def _kept_points(energy, pol, x):
    """Return a boolean keep-mask for reflectivity points.

    Drops the 284.2 eV fringe bottoms (three p-pol points near 0.0125 inverse
    angstrom and four s-pol points near 0.0375), the minimum p-pol point at
    285.1 eV and at 287 eV, and any point at exactly zero momentum transfer.

    Parameters
    ----------
    energy : float
        Photon energy in eV.
    pol : {"s", "p"}
        Polarization of ``x``.
    x : array of float
        Momentum transfer in inverse angstrom.

    Returns
    -------
    numpy.ndarray of bool
        True where the point is drawn and enters the within-2-sigma count.
    """
    q_drop = _MASK_Q.get((float(energy), pol))
    keep = np.ones(np.shape(x)[0], dtype=bool)
    if q_drop is None:
        return keep
    x_arr = np.asarray(x, dtype=float)
    for q0 in q_drop:
        keep &= ~np.isclose(x_arr, q0, rtol=0.0, atol=5e-4)
    return keep


def _stack_models(energy):
    return {
        "Graded": graded_polish.at(energy).model,
        "DFT slab": objective_view_at(refloxide_objective, energy, swap_pol=False).model,
    }


def _pol_color(base, pol):
    """Return the s or p shade of a data color.

    Parameters
    ----------
    base : color
        Matplotlib color of the dataset.
    pol : {"s", "p"}
        Polarization. ``s`` is darker. ``p`` is lighter. A neutral base stays
        grey.

    Returns
    -------
    tuple of float
        RGB triple in 0-1.
    """
    hue, light, sat = colorsys.rgb_to_hls(*to_rgb(base))
    neutral = sat < 0.05
    if pol == "s":
        light, sat = (0.28, 0.0) if neutral else (0.38, min(1.0, max(sat, 0.90)))
    elif pol == "p":
        light, sat = (0.70, 0.0) if neutral else (0.70, min(sat, 0.38))
    else:
        raise ValueError(f"pol must be 's' or 'p', got {pol!r}")
    return colorsys.hls_to_rgb(hue, light, sat)


def _figure_q_window(energy):
    """Q limits passed to ``_draw_energy`` for this energy.

    Parameters
    ----------
    energy : float
        Photon energy in eV. Must be one of ``STACK_ENERGIES``.

    Returns
    -------
    q_lo, q_hi : float
        Inclusive window in inverse angstroms. 283.7 eV uses the full
        plotted range. The other energies use the short-Q waterfall window.
    """
    if energy == STACK_ENERGIES[0]:
        return MAIN_Q_LO, float(Q.max())
    ds = data[str(energy)]
    q_hi = min(INSET_Q[1], max(float(np.max(ds.s.x)), float(np.max(ds.p.x))))
    return INSET_Q[0], q_hi


def _reduced_chi2(chi2: float, ndata: int, nparams: int) -> float:
    """Return Fig. 2 reduced chi-square, ``chi2 / (ndata - nparams)``.

    Parameters
    ----------
    chi2 : float
        Sum of squared ``logY`` residuals from ``Objective.chisqr``.
    ndata : int
        Number of reflectivity points in the objective.
    nparams : int
        Number of varying parameters. Must be strictly less than ``ndata``.

    Returns
    -------
    float
        Reduced chi-square.

    Raises
    ------
    ValueError
        If ``nparams`` is not strictly less than ``ndata``.
    """
    dof = ndata - nparams
    if dof <= 0:
        raise ValueError(f"non-positive degrees of freedom: N={ndata}, p={nparams}")
    return float(chi2) / float(dof)


def _slab_chi2_on_gradient_points(slab_fit, gradient) -> tuple[float, int]:
    """Sum Fig. 2 slab ``logY`` residuals on the gradient objective's points.

    Parameters
    ----------
    slab_fit
        Fig. 2 DFT ``GlobalObjective`` (``dft_en_offset_new2.pkl``).
    gradient
        Sliced multi-energy ``logY`` objective. Its rows are the comparison
        sample.

    Returns
    -------
    chi2 : float
        Sum of squared ``logY`` residuals of the slab model.
    n : int
        Number of gradient rows matched to a slab point. Matches the
        gradient's ``npoints`` when every gradient row is in the slab data.

    Raises
    ------
    ValueError
        If a gradient row has no slab point within ``1e-6`` in Q.
    """
    rows = gradient._dataset
    energy = np.asarray(rows.energy, dtype=float)
    pol = np.asarray(rows.pol).astype(str)
    q = np.asarray(rows.q, dtype=float)
    chi2 = 0.0
    matched = 0
    for obj in slab_fit.objectives:
        obj_energy = float(obj.model.energy)
        if not np.any(np.isclose(energy, obj_energy, rtol=0.0, atol=1e-3)):
            continue
        residuals = np.asarray(obj.residuals(), dtype=float)
        n_s = len(obj.data.s.x)
        for name, span in (("s", slice(0, n_s)), ("p", slice(n_s, None))):
            x = np.asarray(getattr(obj.data, name).x, dtype=float)
            keep = np.zeros(x.size, dtype=bool)
            wanted = q[np.isclose(energy, obj_energy, rtol=0.0, atol=1e-3) & (pol == name)]
            for i, qi in enumerate(x):
                keep[i] = bool(np.any(np.isclose(wanted, qi, rtol=0.0, atol=1e-6)))
            chi2 += float(np.sum(residuals[span][keep] ** 2))
            matched += int(keep.sum())
    if matched != int(gradient.npoints):
        raise ValueError(
            f"gradient points missing from the slab data: matched {matched} "
            f"of {int(gradient.npoints)}"
        )
    return chi2, matched


def _selection_criteria():
    """Reduced chi-square of each model on the gradient fit's points.

    ``chi2`` is the sum of squared residuals in the ``logY`` transform.
    The divisor is ``N - n_vary`` with each model's own varying-parameter
    count, as in Fig. 2. ``N`` is the sliced gradient objective for both
    models, not the slab's 21-energy sample.

    Returns
    -------
    dict
        Model name to reduced chi-square.
    """
    with graded_polish.path.open("rb") as handle:
        gradient = pickle.load(handle)["gpu_objective"]
    g_chi2 = float(gradient.chisqr())
    g_n = int(gradient.npoints)
    g_p = len(gradient.varying_parameters())
    graded = _reduced_chi2(g_chi2, g_n, g_p)

    slab_path = models_root / "xrr/znpc/dft/dft_en_offset_new2.pkl"
    with slab_path.open("rb") as handle:
        slab_fit = pickle.load(handle)
    s_chi2, s_n = _slab_chi2_on_gradient_points(slab_fit, gradient)
    s_p = len(slab_fit.varying_parameters())
    slab = _reduced_chi2(s_chi2, s_n, s_p)

    chi_red = {"Graded": graded, "DFT slab": slab}
    print(
        f"Gradient N={g_n} p={g_p} chi2={g_chi2:.2f} chi2_nu={graded:.4f}"
    )
    print(f"Slab N={s_n} p={s_p} chi2={s_chi2:.2f} chi2_nu={slab:.4f}")
    return chi_red


def _place_chi_labels(ax, legend, chi_red, pol_legend):
    """Place each reduced chi-square under its legend label, inside the axes.

    Values are rounded to one decimal place. The font matches the tick
    labels. If a string would cross the right spine, the model legend, the
    polarization legend, and both strings shift left together.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes that owns ``legend``.
    legend : matplotlib.legend.Legend
        Legend whose texts follow ``STACK_MODELS`` order.
    chi_red : dict
        Model name to Fig. 2 reduced chi-square.
    pol_legend : matplotlib.legend.Legend
        Polarization legend stacked above ``legend``. Shifted with it.
    """
    fontsize = float(plt.rcParams["xtick.labelsize"])
    colors = {mdl["name"]: mdl["color"] for mdl in STACK_MODELS}
    names = [mdl["name"] for mdl in STACK_MODELS]
    pad_px = 6.0

    def _legend_centers(renderer):
        frame = legend.get_frame().get_window_extent(renderer=renderer)
        _, y_frame = ax.transAxes.inverted().transform((0.0, frame.y0))
        centers = []
        for text in legend.get_texts():
            bb = text.get_window_extent(renderer=renderer)
            (x0, _), (x1, _) = ax.transAxes.inverted().transform(
                [(bb.x0, bb.y0), (bb.x1, bb.y1)]
            )
            centers.append(0.5 * (x0 + x1))
        return y_frame - 0.012, centers

    def _stamp(renderer):
        y, centers = _legend_centers(renderer)
        artists = []
        for name, xc in zip(names, centers, strict=True):
            artists.append(
                ax.text(
                    xc,
                    y,
                    rf"$\chi^2_\nu = {chi_red[name]:.1f}$",
                    transform=ax.transAxes,
                    ha="center",
                    va="top",
                    fontsize=fontsize,
                    color=colors[name],
                    clip_on=False,
                    zorder=10,
                )
            )
        return artists

    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    artists = _stamp(renderer)
    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    ax_bb = ax.get_window_extent(renderer=renderer)
    right = max(art.get_window_extent(renderer=renderer).x1 for art in artists)
    overflow = right - (ax_bb.x1 - pad_px)
    if overflow > 0.0:
        origin = ax.transAxes.inverted().transform((0.0, 0.0))
        shifted = ax.transAxes.inverted().transform((overflow, 0.0))
        dx = float(shifted[0] - origin[0])
        anchor = legend.get_bbox_to_anchor()
        pol_anchor = pol_legend.get_bbox_to_anchor()
        legend.set_bbox_to_anchor((float(anchor.x0) - dx, float(anchor.y0)))
        pol_legend.set_bbox_to_anchor(
            (float(pol_anchor.x0) - dx, float(pol_anchor.y0))
        )
        for art in artists:
            art.remove()
        ax.figure.canvas.draw()
        renderer = ax.figure.canvas.get_renderer()
        artists = _stamp(renderer)
        ax.figure.canvas.draw()
        renderer = ax.figure.canvas.get_renderer()
    boxes = [art.get_window_extent(renderer=renderer) for art in artists]
    print(
        "rmse axes x: "
        + ", ".join(
            f"{names[i]} "
            f"{ax.transAxes.inverted().transform((bb.x0, bb.y0))[0]:.3f}-"
            f"{ax.transAxes.inverted().transform((bb.x1, bb.y0))[0]:.3f}"
            for i, bb in enumerate(boxes)
        )
    )


def _draw_energy(ax, energy, q_lo, q_hi, *, shift=1.0, ms=DATA_MS, lw_scale=1.0,
                 label_models=False, every=None, q_scale=1.0):
    """Data and both models for one energy over [q_lo, q_hi], multiplied by shift.

    ``q_scale`` stretches the abscissa for zoomed short-Q waterfall traces.
    """
    ds = data[str(energy)]
    models = _stack_models(energy)
    q = np.linspace(q_lo, q_hi, 1200)
    for pol, marker in (("s", "o"), ("p", "^")):
        d = getattr(ds, pol)
        x = np.asarray(d.x)
        m = (x >= q_lo) & (x <= q_hi) & _kept_points(energy, pol, x)
        xs, ys, es = _subsample_xyerr(x[m], d.y[m], d.y_err[m],
                                      DATA_EVERY if every is None else every)
        col = _pol_color(C_S, pol)
        ax.errorbar(xs * q_scale, ys * shift, yerr=es * shift, fmt="none",
                    elinewidth=0.6, capsize=1.15, capthick=1, ecolor=col,
                    zorder=2)
        ax.plot(xs * q_scale, ys * shift, linestyle="none", marker=marker,
                ms=ms, color=col, zorder=3)
        for mdl in STACK_MODELS:
            ax.plot(q * q_scale,
                    np.asarray(rxr(q, models[mdl["name"]], pol), dtype=float) * shift,
                    color=mdl["color"], ls=mdl["ls"], lw=.75,
                    zorder=6 if mdl["name"] == "Graded" else 10,
                    label=STACK_LABELS.get(mdl["name"], mdl["name"])
                    if (label_models and pol == "s") else None)
    return q, models


def _trace_envelope(energy, q_lo, q_hi, grid):
    """Lower and upper log10 envelope of everything drawn for one energy.

    Data (both polarizations) and both models, on a common Q grid; NaN where
    the energy has no data, so the exclusion only acts where curves coexist.
    """
    ds = data[str(energy)]
    models = _stack_models(energy)
    traces = []
    inside = (grid >= q_lo) & (grid <= q_hi)
    for pol in ("s", "p"):
        d = getattr(ds, pol)
        x = np.asarray(d.x, dtype=float)
        m = (x >= q_lo) & (x <= q_hi) & (np.asarray(d.y) > 0) & _kept_points(energy, pol, x)
        o = np.argsort(x[m])
        yd = np.interp(grid, x[m][o], np.log10(np.asarray(d.y)[m][o]),
                       left=np.nan, right=np.nan)
        traces.append(np.where(inside, yd, np.nan))
        for mdl in STACK_MODELS:
            ym = np.log10(np.clip(np.asarray(rxr(grid, models[mdl["name"]], pol),
                                             dtype=float), 1e-30, None))
            traces.append(np.where(inside, ym, np.nan))
    traces = np.vstack(traces)
    with warnings.catch_warnings():  # columns past q_hi are all NaN by design
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmin(traces, axis=0), np.nanmax(traces, axis=0)


def inset_offsets(q_his):
    """Exclusion stacking. Each energy is lowered just enough that its upper
    envelope clears the lower envelope of the energy above it by INSET_GAP_DEC
    at every Q they share, so curves never cross and a flat curve (283.7 eV)
    takes only the height it needs. Below INSET_LABEL_Q the clearance grows by
    INSET_LABEL_GAP_DEC, the room the energy tag over each first point needs.
    Returns multiplicative shifts."""
    grid = np.linspace(INSET_Q[0], INSET_Q[1], 600)
    env = [_trace_envelope(e, INSET_Q[0], qh, grid) for e, qh in zip(INSET_ENERGIES, q_his)]
    extra = INSET_LABEL_GAP_DEC * (grid < INSET_LABEL_Q)
    offs = [0.0]
    for k in range(1, len(env)):
        lo_prev = env[k - 1][0] + offs[k - 1]
        hi_k = env[k][1]
        clear = np.nanmin(lo_prev - hi_k - extra)
        offs.append(float(clear - INSET_GAP_DEC))
    return [10.0 ** o for o in offs]


def energy_abs_chi(energy):
    """|chi| of every point of one energy, both polarizations, full Q range.

    Returns ``{model name: array}`` for the models in ``STACK_MODELS``; ``chi``
    follows ``RESIDUAL_KIND`` exactly as in the residual strip.
    """
    ds = data[str(energy)]
    models = _stack_models(energy)
    out = {}
    for mdl in STACK_MODELS:
        chis = []
        for pol in ("s", "p"):
            d = getattr(ds, pol)
            x = np.asarray(d.x, dtype=float)
            r_d = np.asarray(d.y, dtype=float)
            sig = np.asarray(d.y_err, dtype=float)
            r_m = np.asarray(rxr(x, models[mdl["name"]], pol), dtype=float)
            ok = (
                np.isfinite(r_m) & (sig > 0) & (r_d > 0) & _kept_points(energy, pol, x)
            )
            chis.append(
                ((r_d - r_m) / sig)[ok] if RESIDUAL_KIND == "linear"
                else log_residual(r_d[ok], r_m[ok], sig[ok])
            )
        out[mdl["name"]] = np.abs(np.concatenate(chis))
    return out


def draw_rmse_bars(ax):
    """Goodness-of-fit inset in the bottom-right hole of panel (a).

    Sits right of the zoomed short-Q waterfall and beneath the 283.7 eV high-Q
    tail. Vertical bars (not rotated): pooled within-``FIT_SIGMA`` performance
    and varying-parameter count for Gradient vs Slab.
    """
    fs = plt.rcParams["xtick.labelsize"]
    xlim = ax.get_xlim()
    q_lo = INSET_Q[1] * STACK_Q_SCALE * 1.20
    x0 = float(np.clip((q_lo - xlim[0]) / (xlim[1] - xlim[0]), 0.58, 0.72))
    w = float(np.clip(0.985 - x0, 0.28, 0.40))
    y0, h = 0.05, 0.40
    print(f"rmse inset axes-frac: ({x0:.2f}, {y0:.2f}, {w:.2f}, {h:.2f})")

    gap = 0.07 * w
    w_b = 0.60 * (w - gap)
    w_p = (w - gap) - w_b
    axB = ax.inset_axes((x0, y0, w_b, h))
    axP = ax.inset_axes((x0 + w_b + gap, y0, w_p, h))
    per_e = {e: energy_abs_chi(e) for e in STACK_ENERGIES}
    pooled = {m["name"]: np.concatenate([per_e[e][m["name"]] for e in STACK_ENERGIES])
              for m in STACK_MODELS}
    pct = {m["name"]: 100.0 * float(np.mean(pooled[m["name"]] < FIT_SIGMA))
           for m in STACK_MODELS}
    print(f"% within {FIT_SIGMA:g} sigma all (gradient/slab): "
          + "/".join(f"{pct[m['name']]:.0f}" for m in STACK_MODELS))

    for j, mdl in enumerate(STACK_MODELS):
        dx = (j - 0.5) * 0.40
        axB.bar(dx, pct[mdl["name"]], 0.38, color=mdl["color"], lw=0, zorder=3)
        axB.text(dx, pct[mdl["name"]] + 3.0, f"{pct[mdl['name']]:.0f}",
                 ha="center", va="bottom", fontsize=fs, color=mdl["color"],
                 clip_on=False, zorder=6)
        n = N_PARAMS[mdl["name"]]
        axP.bar(dx, n, 0.38, color=mdl["color"], lw=0, zorder=3)
        axP.text(dx, n + 0.6, f"{n:d}", ha="center", va="bottom", fontsize=fs,
                 color=mdl["color"], clip_on=False, zorder=6)

    axB.set_ylim(10.0, 90.0)
    axB.set_xlim(-0.55, 0.55)
    axB.set_xticks([0.0])
    axB.set_xticklabels([rf"within {FIT_SIGMA:g}$\sigma$"])
    axB.set_yticks([25, 50, 75])
    axB.set_yticklabels(["25%", "50%", "75%"])
    axP.set_ylim(0.0, 32.0)
    axP.set_xlim(-0.55, 0.55)
    axP.set_xticks([0.0])
    axP.set_xticklabels([r"$N$"])
    axP.set_yticks([0, 10, 20])
    for a_ in (axB, axP):
        a_.set_zorder(ax.get_zorder() + 1)
        a_.patch.set_facecolor("white")
        a_.patch.set_alpha(1.0)
        a_.tick_params(which="both", labelsize=fs, pad=1.5, length=2)
        a_.xaxis.set_tick_params(which="both", top=False)
        a_.xaxis.set_minor_locator(NullLocator())
        a_.yaxis.set_minor_locator(NullLocator())
        a_.spines["top"].set_visible(False)
        a_.grid(False)
        for t in list(a_.get_yticklabels()) + list(a_.get_xticklabels()):
            t.set_clip_on(False)
    axB.spines["right"].set_visible(False)
    axB.yaxis.set_tick_params(which="both", right=False)
    for lab in axB.get_yticklabels():
        lab.set_rotation(90)
        lab.set_ha("center")
        lab.set_va("center")
        lab.set_rotation_mode("anchor")
    axB.tick_params(axis="y", pad=6)
    axP.spines["left"].set_visible(False)
    axP.yaxis.tick_right()
    axP.yaxis.set_tick_params(which="both", left=False, labelright=False)
    d, lw = 4.0, plt.rcParams["axes.linewidth"]
    for a_, xe in ((axB, 1.0), (axP, 0.0)):
        a_.plot([xe], [0.0], transform=a_.transAxes, marker=[(-1, -1), (1, 1)],
                markersize=d, linestyle="none",
                color=plt.rcParams["axes.edgecolor"],
                mec=plt.rcParams["axes.edgecolor"], mew=lw, clip_on=False, zorder=7)
    return axB


def _label_energy(ax, energy, q_lo, q_hi, shift, q_scale, q_at):
    """Write the photon energy just above one waterfall trace.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Reflectivity axes. Must already use a logarithmic y scale.
    energy : float
        Photon energy in eV.
    q_lo, q_hi : float
        Physical momentum-transfer window of the trace, in inverse angstrom.
    shift : float
        Multiplicative vertical offset of the trace.
    q_scale : float
        Abscissa stretch applied to zoomed short-Q traces.
    q_at : float
        Physical Q, in inverse angstrom, where the label is anchored.
    """
    d = data[str(energy)].s
    x = np.asarray(d.x, dtype=float)
    y = np.asarray(d.y, dtype=float)
    m = (x >= q_lo) & (x <= q_hi) & (y > 0) & _kept_points(energy, "s", x)
    if not np.any(m):
        return
    xs = x[m]
    ys = y[m]
    i = int(np.argmin(np.abs(xs - q_at)))


def _place_pol_legend(ax, model_legend):
    """Place the s and p markers above the model legend and drop that legend.

    ``s`` is the darker circle and ``p`` is the lighter triangle, matching
    ``_draw_energy``. The polarization frame keeps the model legend's top
    edge. The model legend, anchored at its upper right, moves down by the
    polarization frame height so the stack reads s/p, then Gradient/Slab.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Reflectivity axes.
    model_legend : matplotlib.legend.Legend
        Gradient and slab legend already drawn on ``ax``.

    Returns
    -------
    matplotlib.legend.Legend
        The polarization legend.
    """
    ms = 4.5
    handles = []
    for pol, marker, label in (("s", "o", "$s$-pol"), ("p", "^", "$p$-pol")):
        color = _pol_color(C_S, pol)
        handles.append(
            Line2D(
                [0], [0], linestyle="none", marker=marker, ms=ms,
                markerfacecolor=color, markeredgecolor=color, label=label,
            )
        )
    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    frame = model_legend.get_frame().get_window_extent(renderer=renderer)
    (x0, _), (x1, y_top) = ax.transAxes.inverted().transform(
        [(frame.x0, frame.y1), (frame.x1, frame.y1)]
    )
    ax.add_artist(model_legend)
    pol = ax.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5 * (x0 + x1), y_top),
        ncols=2,
        handlelength=0.9,
        handletextpad=0.35,
        columnspacing=1.4,
        borderpad=0.25,
        borderaxespad=0.0,
        framealpha=0.95,
        edgecolor="0.75",
        fancybox=False,
        fontsize=float(plt.rcParams["xtick.labelsize"]),
    )
    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    pol_frame = pol.get_frame().get_window_extent(renderer=renderer)
    _, y_pol = ax.transAxes.inverted().transform((0.0, pol_frame.y0))
    model_legend.set_bbox_to_anchor((x1, y_pol - 0.006))
    return pol


def draw_reflectivity_panel(ax):
    """283.7 eV full-Q; short-Q energies as a zoomed waterfall; fit metrics in the gap."""
    e0 = STACK_ENERGIES[0]
    chi_red = _selection_criteria()
    _draw_energy(ax, e0, MAIN_Q_LO, float(Q.max()), label_models=True)
    ax.set_yscale("log")
    ax.set_xlim(0.0, float(Q.max()))
    ax.set_ylim(*MAIN_YLIM)
    ax.set_ylabel("Reflectivity (a.u.)")
    ax.set_xlabel(r"$Q\ (\mathrm{\AA}^{-1})$", labelpad=1)
    ax.minorticks_on()
    legend = ax.legend(loc="upper right", bbox_to_anchor=LEGEND_ANCHOR, ncols=2,
                       handlelength=1.4, handletextpad=0.4, columnspacing=2.8,
                       borderpad=0.3, borderaxespad=0.0, framealpha=0.95,
                       edgecolor="0.75", fancybox=False)
    pol = _place_pol_legend(ax, legend)
    _place_chi_labels(ax, legend, chi_red, pol)

    q_his = [min(INSET_Q[1], max(float(np.max(data[str(e)].s.x)),
                                 float(np.max(data[str(e)].p.x))))
             for e in INSET_ENERGIES]
    shifts = [7 ** (-(STACK_TOP_DEC + STACK_DECADES * k))
              for k in range(len(INSET_ENERGIES))]
    print("waterfall offsets (decades): "
          + ", ".join(f"{e:g} {np.log10(sh):+.2f}" for e, sh in zip(INSET_ENERGIES, shifts)))
    y_lo = MAIN_YLIM[0]
    for k, e in enumerate(INSET_ENERGIES):
        shift, q_hi = shifts[k], q_his[k]
        _draw_energy(ax, e, INSET_Q[0], q_hi, shift=shift, every=INSET_EVERY,
                     q_scale=STACK_Q_SCALE)
        for pol in ("s", "p"):
            d = getattr(data[str(e)], pol)
            x = np.asarray(d.x)
            m = (
                (x >= INSET_Q[0]) & (x <= q_hi) & (np.asarray(d.y) > 0)
                & _kept_points(e, pol, x)
            )
            if np.any(m):
                y_lo = min(y_lo, float(np.min(d.y[m]) * shift))
    ax.set_ylim(0.4 * y_lo, MAIN_YLIM[1])
    draw_rmse_bars(ax)
    # ax.text(
    #     0.015, 0.02,
    #     rf"zoomed $\times${STACK_Q_SCALE:g}",
    #     transform=ax.transAxes,
    #     ha="left",
    #     va="bottom",
    #     fontsize=5.0,
    #     color="0.35",
    #     zorder=14,
    #     path_effects=[pe.withStroke(linewidth=1.4, foreground="white")],
    # )
    return ax


# ---- 3. assemble -------------------------------------------------------------
# Depth block: gamma and rho side by side (shared depth axis), MD structure to
# their right. PLATE_SPLIT is depth-block width vs reflectivity.
PLATE_SPLIT = 1.22
FIG_H = 4.35 if HORIZ else 3.6
fig = plt.figure(figsize=(7.55, FIG_H), dpi=300)
outer = fig.add_gridspec(
    1, 3,
    width_ratios=[1.0, 0.17, PLATE_SPLIT],
    wspace=0.0,
    left=0.075, right=0.988, top=0.935,
    bottom=0.115 if HORIZ else 0.125,
)
axR = fig.add_subplot(outer[0])

_keys = list(FILM_ORDER)
if HORIZ:
    _keys = ["structure"] + [k for k in FILM_ORDER if k != "structure"]
    _rows = ([WAVE_WIDTH] if SHOW_WAVE else []) + [FILM_HEIGHT[k] for k in _keys]
    grp = outer[2].subgridspec(len(_rows), 1, height_ratios=_rows, hspace=0.26)
else:
    _cols = [WAVE_WIDTH, WAVE_GAP] if SHOW_WAVE else []
    _slot = {}
    for _k in _keys:
        if _k == "structure":
            _cols.append(STRUCTURE_GAP)
        _slot[_k] = len(_cols)
        _cols.append(FILM_WIDTH[_k])
    grp = outer[2].subgridspec(1, len(_cols), width_ratios=_cols, wspace=0.0)

_i0 = (1 if HORIZ else 2) if SHOW_WAVE else 0
axW = fig.add_subplot(grp[0]) if SHOW_WAVE else None
_share = axW
_film_axes = {}
_own_depth = STRUCTURE_STYLE == "cutout"  # the oblique scene keeps its own depth axis
for _i, _k in enumerate(_keys):
    # the growth strip blanks its own ticks; a shared depth axis would blank the profiles' too
    _alone = (_own_depth or STRUCTURE_STYLE == "growth") and _k == "structure"
    _kw = (
        {("sharex" if HORIZ else "sharey"): _share}
        if _share is not None and not _alone
        else {}
    )
    _ax = fig.add_subplot(grp[_i0 + _i if HORIZ else _slot[_k]], **_kw)
    if _share is None and not _alone:
        _share = _ax
    _film_axes[_k] = _ax
axa, axS, axd = (_film_axes[k] for k in ("gamma", "structure", "rho"))
axS.set_zorder(1)
for _ax in (axa, axd):
    _ax.set_zorder(2)
    if STRUCTURE_STYLE == "growth":  # profile spines draw over the strip on both seams
        _ax.patch.set_visible(False)

draw_reflectivity_panel(axR)

if SHOW_WAVE:
    draw_wave(axW)
draw_orientation_v(axa)
draw_density_v(axd)

_edge = [k for k in _keys if k != "structure"]
_depth_ax = axW if SHOW_WAVE else _film_axes[_edge[-1] if HORIZ else _edge[0]]
for _k, _ax in _film_axes.items():
    if _ax is _depth_ax or _k == "structure":
        continue
    _ax.tick_params(**({"labelbottom": False} if HORIZ else {"labelleft": False}))
(_depth_ax.set_xlabel if HORIZ else _depth_ax.set_ylabel)("depth (nm)")
(_depth_ax.set_xticks if HORIZ else _depth_ax.set_yticks)(
    np.arange(0.0, TOTAL_NM + 0.1, 5.0)
)
tau_tags(_film_axes[_edge[0]])
set_depth_lim(_depth_ax)

axa.set_xticks([30, 50, 70])
axd.set_xlim(0.8, 2)
axd.set_xticks([1, 1.5, 2])
axa.set_yticks([0, 2, 4, 6, 8, 10, 12, 14, 16, 18])

# ---- 4. structure panel: REC_L5c, same depth furniture as gamma and rho ------
import fig4_md_panels as _mdpanels

importlib.reload(_mdpanels)
if STRUCTURE_STYLE == "growth":
    # Molecule strip of the main branch (fig4_panel_b_growth), drawn on a transposed box
    # and moved into place. fig4_panel_b_growth sizes atoms and bonds from the axes width
    # per angstrom of depth, so it is called on a box whose width matches the extent the
    # depth span will finally occupy; point sizes then survive the move back.
    fig.canvas.draw()
    _fw, _fh = fig.get_size_inches()
    _pos = axS.get_position()
    _w_in, _h_in = _pos.width * _fw, _pos.height * _fh
    _span_full = _fig41.DEPTH_XLIM[1] - _fig41.DEPTH_XLIM[0]
    if HORIZ:
        Y_HALF = 0.5 * SPAN_DISP * _h_in / _w_in
        axS.set_position([_pos.x0, _pos.y0, (_w_in * _span_full / SPAN_DISP) / _fw, _h_in / _fh])
    else:
        Y_HALF = 0.5 * _w_in * SPAN_DISP / _h_in
        axS.set_position([_pos.x0, _pos.y0, (_h_in * _span_full / SPAN_DISP) / _fw, _w_in / _fh])
    axS.set_xlim(*_fig41.DEPTH_XLIM)
    axS.set_ylim(-Y_HALF, Y_HALF)
    _fig41.draw_structure(axS, Y_HALF + 0.30 * _fig41.L_MOL)
    axS.set_position(_pos)

    # spacings read from the drawn atoms (front column), still in angstrom
    def _strip_sf(x, q, taper=False, chunk=64):
        x = np.asarray(x, dtype=float)
        w = np.ones_like(x)
        if taper:
            u = (x - x.min()) / max(float(np.ptp(x)), 1e-9)
            w = 0.5 - 0.5 * np.cos(2.0 * np.pi * u)
        out = np.empty_like(q)
        for _i in range(0, q.size, chunk):
            _ph = np.outer(q[_i:_i + chunk], x)
            out[_i:_i + chunk] = (np.cos(_ph) * w).sum(1) ** 2 + (np.sin(_ph) * w).sum(1) ** 2
        return out / max(w.sum(), 1.0) ** 2

    _q = np.linspace(0.25, 2.50, 900)
    _front = [_c for _c in axS.collections if not isinstance(_c, LineCollection) and _c.get_alpha() in (None, 1.0)]
    _xy = np.vstack([_c.get_offsets() for _c in _front])
    _s_lam, _s_col = _strip_sf(_xy[:, 0], _q), _strip_sf(_xy[:, 1], _q, taper=True)
    _win = lambda q0: (_q > q0 - 0.25) & (_q < q0 + 0.25)
    _q_lam = float(_q[_win(0.50)][np.argmax(_s_lam[_win(0.50)])])
    _q_col = float(_q[_win(1.60)][np.argmax(_s_col[_win(1.60)])])
    _md_info = {
        "packing": _fig41.PACKING,
        "d_lam_drawn": round(2 * np.pi / _q_lam, 2),
        "stack_pitch_drawn": round(2 * np.pi / _q_col, 2),
        "pi_spacing": _fig41.PI_SPACING,
        # contact rests on the Si; molecules moved or left out to keep CLASH_A between rings
        "moved_clear": _fig41.LAST_DRAW.get("lifted", 0),
        "vacancies_from_clash": _fig41.LAST_DRAW.get("dropped", 0),
        # interfacial zone, front column: (drawn molecules, expected from rho(z))
        "zone_drawn_vs_rho": _fig41.LAST_DRAW.get("density_zone", {}).get(0),
        "contact_kept": round(getattr(_fig41, "CONTACT_KEEP", {}).get(0, 1.0), 2),
        # cos^2-mean tilt over the first 1.2 nm from the Si: (drawn, fit)
        "contact_band_gamma": _fig41.LAST_DRAW.get("contact_band_gamma"),
    }

    for _txt in list(axS.texts):  # module labels are axes-fraction, wrong after rotation
        _txt.remove()
    _trans = Affine2D().scale(A2NM) if HORIZ else Affine2D().rotate_deg(90).scale(A2NM)
    for _coll in axS.collections:
        if isinstance(_coll, LineCollection):
            _coll.set_transform(_trans + axS.transData)
        else:  # scatter: offsets transform, marker paths are points and must not
            _coll.set_offset_transform(_trans + axS.transData)
    if HORIZ:
        axS.set_xlim(*DEPTH_LIM)
        axS.set_ylim(-Y_HALF * A2NM, Y_HALF * A2NM)
    else:
        axS.set_xlim(-Y_HALF * A2NM, Y_HALF * A2NM)
        axS.set_ylim(*DEPTH_LIM)
    zone_bands(axS)
    tau_rules(axS)
    substrate_bar(axS, label=False)
    axS.set_xticks([])
    axS.set_yticks([])
    axS.tick_params(which="both", bottom=False, left=False, labelbottom=False, labelleft=False)
    for _sp in (("top", "bottom") if HORIZ else ("left", "right")):
        axS.spines[_sp].set_visible(True)
        axS.spines[_sp].set_color("0.6")
elif STRUCTURE_STYLE == "cutout":
    _md_info = _mdstrip.draw_md_structure_3d(
        axS,
        _MD_TAG,
        total_thick=TOTAL,
        a2nm=A2NM,
        depth_lim_nm=DEPTH_LIM,
        slab_y=20.0,
        azim=8.0,
        elev=6.0,
        sub_nm=SUB_NM,
        tau_vac_nm=TAU_VAC_NM,
        tau_si_nm=TAU_SI_NM,
        label_tau=False,
    )
    axS.set_axis_on()
elif STRUCTURE_STYLE == "wire":
    zone_bands(axS)
    tau_rules(axS)
    substrate_bar(axS, label=True, zorder=1.2)
    _md_info = _mdpanels.draw_wire(
        axS,
        _MD_TAG,
        total_thick=TOTAL,
        depth_lim_nm=DEPTH_LIM,
        elev=3.0,
        near_alpha=0.55,
        far_alpha=0.04,
        linewidth=0.25,
    )
elif STRUCTURE_STYLE == "ticks":
    zone_bands(axS)
    tau_rules(axS)
    substrate_bar(axS, label=True, zorder=1.2)
    _md_info = _mdpanels.draw_cell_ticks(
        axS,
        total_thick=TOTAL,
        depth_lim_nm=DEPTH_LIM,
        z_a=Z_G,
        gamma_deg=ALPHA_ROUGH,
        rho_sharp=RHO_SHARP,
        tau_si=TAU_SI,
        tau_vac=TAU_VAC,
        surface_roughness=float(film.surface_roughness.value),
    )
else:
    raise ValueError(
        f"STRUCTURE_STYLE must be 'growth', 'cutout', 'ticks' or 'wire', got {STRUCTURE_STYLE!r}"
    )
if STRUCTURE_STYLE != "growth":
    set_depth_lim(axS)
    axS.set_xticks([])
    axS.xaxis.set_minor_locator(NullLocator())
    axS.minorticks_on()
    axS.xaxis.set_minor_locator(NullLocator())
if _own_depth:
    _in_film = lambda t: [v for v in t if 0.0 <= v <= TOTAL_NM]
    axS.set_yticks(_in_film(axa.get_yticks()))
    axS.set_yticks(_in_film(axa.get_yticks(minor=True)), minor=True)
if STRUCTURE_STYLE != "growth":
    axS.tick_params(axis="y", labelleft=False)
axS.grid(False)
if _own_depth:
    axS.patch.set_visible(False)
    for _sp in ("top", "right", "bottom"):
        axS.spines[_sp].set_visible(False)
    axS.spines["left"].set_position(("outward", STRUCTURE_AXIS_OFFSET_PT))
    axS.spines["left"].set_bounds(0.0, TOTAL_NM)
    axS.tick_params(axis="y", which="both", direction="out", right=False, left=True)
print(f"structure panel: {_MD_TAG} {STRUCTURE_STYLE} {_md_info}")
if STRUCTURE_STYLE == "growth":
    # lying contact islands (front column): sub-Bragg facets at the Si
    _isl = [i for i in _fig41.LAST_DRAW.get("islands", []) if i["n"] >= 2]
    print("contact islands (n >= 2): n molecules, height of atom centres, top above mean Si, width (A)")
    for _k, _i in enumerate(_isl, 1):
        print(f"  {_k}: n={_i['n']}  height {_i['height_A']:.1f}  top {_i['top_above_Si_A']:.1f}  width {_i['width_A']:.0f}")
    if _isl:
        _h = np.array([_i["height_A"] for _i in _isl])
        print(f"  mean height {_h.mean():.1f} A (range {_h.min():.1f} to {_h.max():.1f}); "
              f"+ ~1.3 A atom diameter for the van der Waals extent; d200 = 12.4 A")

# ---- 5. panel letters -------------------------------------------------------
PANEL_LETTERS = "abcdefgh"
_stack = ([axW] if SHOW_WAVE else []) + list(_film_axes.values())
_stack.sort(key=(lambda a: -a.get_position().y1) if HORIZ
            else (lambda a: a.get_position().x0))
_labelled = [(axR, -38.0), *((_a, 1.0) for _a in _stack)]
for _tag, (_ax, _dx) in zip(PANEL_LETTERS, _labelled):
    _ax.annotate(
        f"({_tag})",
        xy=(0.0, 1.0),
        xycoords="axes fraction",
        xytext=(_dx, 3.0),
        textcoords="offset points",
        va="bottom",
        ha="left",
        annotation_clip=False,
    )
SAVENAME = "FIG4"
fig.savefig(f"{SAVENAME}.pdf", dpi=900, transparent=True)
fig.savefig(f"{SAVENAME}.png", dpi=900, transparent=True)
fig


In [ ]:
print(graded_obj.parameters)


## 6. SI 12: GIWAXS vs simulated structure

Compares the SI2-reduced GIWAXS cuts and peak spacings with the tick drawing
from FIG4 panel (d). Requires the Figure 4 cells above so ``Z_G``, ``ALPHA_ROUGH``,
``RHO_SHARP``, ``TOTAL``, ``TAU_SI``, ``TAU_VAC``, and ``DEPTH_LIM`` are in scope.


In [ ]:
import importlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

import fig4_md_panels as _mdpanels
import giwaxs_style as gws   # the SI2 panel style, beside this notebook

importlib.reload(_mdpanels)
importlib.reload(gws)

SAVENAME_SI = "SI12"
_SI2_DIR = Path(_mdpanels.__file__).resolve().parent   # notebooks/manuscript, independent of the kernel cwd
D_PI_FLOOR = 3.0                       # A, below this a drawn pi spacing is not physical

_cuts = pd.read_csv(_SI2_DIR / "SI2_cuts.csv")
_meas = json.loads((_SI2_DIR / "SI2_summary.json").read_text())["comparison"]
_peaks = pd.read_csv(_SI2_DIR / "SI2_peaks.csv").set_index("peak")

# ---- model, row by row ------------------------------------------------------
_rows = _mdpanels.tick_spacings(
    total_thick=TOTAL,
    depth_lim_nm=DEPTH_LIM,
    z_a=Z_G,
    gamma_deg=ALPHA_ROUGH,
    rho_sharp=RHO_SHARP,
    tau_si=TAU_SI,
    tau_vac=TAU_VAC,
)


def _model_peak(d, fwhm, sigma_pos, n_iter=200):
    """Mode, 16/50/84% quantiles in q, and the weights, for one spacing family."""
    q = 2.0 * np.pi / d
    w = _rows["molecules_per_a"] * np.exp(-(q * sigma_pos) ** 2)
    w = w / w.sum()
    s = fwhm / 2.3548
    x = float(np.sum(w * q))
    for _ in range(n_iter):
        k = w * np.exp(-0.5 * ((q - x) / s) ** 2)
        x = float(np.sum(k * q) / np.sum(k))
    o = np.argsort(q)
    cdf = np.cumsum(w[o])
    qs = np.interp([0.16, 0.50, 0.84], cdf, q[o])
    return x, qs, w


_model = {}
for _key in ("d_lam", "d_pi"):
    _x, _qs, _w = _model_peak(_rows[_key], _meas[_key]["fwhm"], _rows["sigma_pos"])
    _model[_key] = dict(q=_x, d=2 * np.pi / _x, q16=_qs[0], q50=_qs[1], q84=_qs[2], w=_w)
    _model[_key]["offset"] = 100.0 * (_model[_key]["d"] / _meas[_key]["d"] - 1.0)
    # offsets of the 16 and 84% quantiles, as spacings
    _model[_key]["off_lo"] = 100.0 * (2 * np.pi / _qs[2] / _meas[_key]["d"] - 1.0)
    _model[_key]["off_hi"] = 100.0 * (2 * np.pi / _qs[0] / _meas[_key]["d"] - 1.0)

_low = _rows["d_pi"] < D_PI_FLOOR
print("=" * 72)
print(f"SI12. measured (SI2, ZnPC_Harlan_0_00000) against the panel (d) drawing, {len(_rows['z'])} rows")
print(f"  cell      d200 {_rows['d200']:.3f} A   pi spacing {_rows['pi_spacing']:.3f} A   n_b {_rows['n_b']:.3f}")
print(f"  {'':8s}{'measured d':>12s}{'model d':>10s}{'offset':>9s}{'model 16-84% (A)':>20s}")
for _key, _lab in (("d_lam", "d_lam"), ("d_pi", "d_pi")):
    _m, _p = _meas[_key], _model[_key]
    print(f"  {_lab:8s}{_m['d']:9.3f} A {_p['d']:8.3f} A {_p['offset']:+7.1f}%"
          f"   {2*np.pi/_p['q84']:6.2f} to {2*np.pi/_p['q16']:5.2f}")
print(f"  rows with d_pi < {D_PI_FLOOR} A: {int(_low.sum())} of {_low.size}, "
      f"{100 * _model['d_pi']['w'][_low].sum():.1f}% of the pi-pi scattering weight, "
      f"minimum {_rows['d_pi'].min():.2f} A at gamma {_rows['gamma'][np.argmin(_rows['d_pi'])]:.0f} deg")
print("=" * 72)

# ---- figure, single column, panel (a) drawn exactly as SI2 (b) ------------------
_fp = _peaks
_CUT_LABELS = {"(200)": ("oop_smooth", _fp.q["(200)"]), "(002)": ("ip_smooth", _fp.q["IP ring, one peak"], 1.16),
               "(400)": ("oop_smooth", _fp.q["(400), one peak"]),
               r"$\mathbf{(\pi-\pi)}$": ("ip_smooth", _fp.q["π-π"], 0.95)}
_rc = plt.rc_context(gws.RC)
_rc.__enter__()
fig = plt.figure(figsize=(3.5, 3.5))
axA = fig.add_axes([0.165, 0.43, 0.81, 0.526])   # same size in inches as SI2 (b)
axB = fig.add_axes([0.165, 0.115, 0.81, 0.155])
gws.draw_cut_panel(axA, _cuts, _CUT_LABELS)

# model positions, lamellar orders 1 and 2 and pi-pi, with their row spread
for _key, _c, _orders in (("d_lam", gws.C_OOP, (1, 2)), ("d_pi", gws.C_IP, (1,))):
    _p = _model[_key]
    for _n in _orders:
        axA.axvspan(_n * _p["q16"], _n * _p["q84"], color=_c, alpha=0.12, lw=0, zorder=1)
        axA.axvline(_n * _p["q"], color=_c, lw=1.0, ls=(0, (3, 2)), zorder=2)

# panel b, model minus measurement as spacing offsets
_ylab = {"d_lam": r"$d_\mathrm{lam}$", "d_pi": r"$d_{\pi}$"}
_span = max(4.0, 1.25 * max(abs(_model[k]["offset"]) for k in _model))
axB.axvline(0.0, color="0.2", lw=0.7, zorder=1)
for _yy, (_key, _c) in zip((1, 0), (("d_lam", gws.C_OOP), ("d_pi", gws.C_IP))):
    _p = _model[_key]
    _lo, _hi = max(_p["off_lo"], -_span), min(_p["off_hi"], _span)
    axB.plot([_lo, _hi], [_yy, _yy], color=_c, lw=4.0, alpha=0.25, solid_capstyle="butt", zorder=2)
    axB.plot([0, _p["offset"]], [_yy, _yy], color=_c, lw=1.2, solid_capstyle="butt", zorder=3)
    axB.plot([_p["offset"]], [_yy], "o", color=_c, ms=4.0, zorder=4)
    axB.annotate(f"{_p['offset']:+.1f}%", xy=(_p["offset"], _yy), xytext=(0, 4), textcoords="offset points",
                 ha="center", va="bottom", fontsize=gws.FS_TICK, color=_c)
axB.set_yticks([1, 0])
axB.set_yticklabels([_ylab["d_lam"], _ylab["d_pi"]])
axB.set_ylim(-0.5, 1.75)
axB.set_xlim(-_span, _span)
axB.set_xlabel("Model offset from measured spacing (%)")
axB.tick_params(axis="y", which="both", left=False, right=False)
axB.grid(False, axis="y")
axB.minorticks_off()

axB.tick_params(axis="x", which="major", length=5, width=1.0)
fig.text(0.015, 0.995, "(a)", va="top", fontsize=gws.FS_LABEL + 1)
fig.text(0.015, 0.33, "(b)", va="top", fontsize=gws.FS_LABEL + 1)
fig.savefig(_SI2_DIR / f"{SAVENAME_SI}.pdf", dpi=600)
fig.savefig(_SI2_DIR / f"{SAVENAME_SI}.png", dpi=600)
print(f"wrote {_SI2_DIR / SAVENAME_SI}.png and .pdf (kernel cwd {Path.cwd()})")
plt.show()
_rc.__exit__(None, None, None)

# kept for the closing comparison with the visualization
GIWAXS_MEASURED = _meas
GIWAXS_MODEL = {k: {kk: v for kk, v in p.items() if kk != "w"} for k, p in _model.items()}
